# EMBER end to end, multilingual Gambling, on Llama-3.1-8B-Instruct: §3.1 Sparse MF → §3.2 concept features → §3.3 embedding erasure

Paper: *Don't Forget Your Embeddings: Robust Knowledge Erasure via Precise Editing of Embeddings*
(Suslik, Shafran, Geva, EMNLP Findings 2026). This is the **multilingual (8-language) variant** of
`ember_three_steps_llama.ipynb`: the same three stages for "Gambling", but $V^*$ is built from sentences in
English, Spanish, French, German, Italian, Portuguese, Hindi and Thai.

- $S_C$: up to 40 Wikipedia Gambling sentences per language (268 in total: en 40, es 40, fr 40, de 40, it 40, pt 32, hi 26, th 10), from `experiments/multilingual_hp/data/concepts/Gambling/concept_sentences.json`.
  Short articles give fewer sentences in some languages; every per-language table handles that.
- $S_N$: 40 neutral sentences per language (320), Gemini translations of English Wikipedia-dump sentences
  (`NEUTRAL_SOURCE = "translated"`, the only source for this concept). `NEUTRAL_REP` (0, 1 or 2) picks one of three
  independently sampled neutral replicates, shared by all concepts (`splitting.neutral_paths`).

The judge sees each token together with the word and language it came from (3.2.a′), and 3.2.g checks whether
the selected concept direction is shared across languages. MC evaluation and the 3.4 $\delta$ tuning stay English-only.
Each stage is split into small blocks, and each block starts with a note on what it does.

**Runs and resume.** Everything expensive is cached in `RUN_DIR = runs/<concept>/translated_rep<NEUTRAL_REP>/`:
the Sparse MF fit (`smf.pt`), every Gemini answer (`judge.json`), MC accuracies (`mc.json`) and generations (`gen.json`),
so a restarted kernel redoes none of it. `SMOKE = True` makes a quick end-to-end pass (50 MF iterations, 3 judged
features, 2 MC items per set, 10-token answers, $\delta \in$ {1, 10}, 30 extended sentences per language and side)
into `RUN_DIR` + `_smoke`. `scripts/run_overnight.py` runs the notebook headless through §3.3 and skips the cells
tagged `skip-overnight` (§3.4).

| Step | Paper section | What happens | Repo code reused |
|---|---|---|---|
| **3.1** | Finding Embedding Features with Sparse MF | factorize $E_{V^*}^\top \approx ZY$ with WTA-sparse $Y$ | `SparseMatrixFactorization`, `fit_with_ridge` |
| **3.2** | Identifying Concept-Related Features | mass ratio $\rho_i \ge \tau$, then a two-stage Gemini judge → $F_C$ | `compute_embedding_stats`, prompts from `ember/interpret_features.py` |
| **3.3** | Erasing Concept-Related Features | $\mathbf e_t \leftarrow \mathbf e_t - \delta \sum_{i\in F_C} Y_{i,t}\,\mathbf z_i$ for concept tokens | same math as `ember/erasure/embed_edit.py` |

**Notation.** Paper $Z \in \mathbb R^{d\times k}$ = code `F_`; paper $Y \in \mathbb R^{k\times|V^*|}$ = code `G_.T`.
In this notebook, `Z` and `Y` use the paper's shapes.

**Requirements:** a GPU, `HF_TOKEN` with access to the gated `meta-llama/Llama-3.1-8B-Instruct`, and a Gemini API key for Step 3.2.

**What changes from the Gemma notebook** (`ember_three_steps.ipynb`), per the paper:

| | Gemma-2-2B-it | Llama-3.1-8B-Instruct | Source |
|---|---|---|---|
| $d$ | 2304 | 4096 (≈1.8× larger) | App. A.1 |
| rank $k$ | 100 | **200** (scaled with $d$; $s$, $\lambda$, $\|S\|$, $\tau$ unchanged) | App. A.1 |
| embedding scale | $\sqrt d$ applied in the forward pass (TL folds it into `W_E`) | **none**: TL `W_E` = HF `embed_tokens` | `embed_edit._embedding_scale` |
| $E$ / $U$ | tied, so `load_hf_model` unties `lm_head` | **untied** already; only $E$ is edited, as in the paper | §6 |
| $\delta$ behaviour | efficacy rises fast up to $\delta\approx 50$ | efficacy saturates at $\delta\approx 50$–$100$; specificity drops sharply from $\delta\approx 50$ | App. C.4, Fig. 11 |
| edited tokens | at most 31–86 (≤0.034% of $V$) | at most 22–174 (≤0.136% of $V$) | §5.2 |

Because the model is 4× larger, this notebook also loads the TransformerLens copy in bf16 (fp32 would need ~32 GB plus the HF copy TL loads first), and the `erased` / `tuned` models share every tensor except the input embedding matrix with `original_model` (≈1 GB each instead of ≈16 GB).

---
## 0. Setup

### 0.1 Hyperparameters (paper defaults)

| Symbol | Value | Source |
|---|---|---|
| $k$ (rank) | 100 for Gemma-2-2B-it, 200 for Llama-3.1-8B-Instruct | App. A.1 |
| $s$ (WTA sparsity on $Y$) | 0.01, so each factor keeps $\lceil 0.01\,|V^*| \rceil$ tokens | App. A.1 |
| $\lambda$ (ridge) | $10^{-4}$ | App. A.1 |
| $T$ / patience / tol | 20,000 iters / 500 iters / $10^{-4}$ | App. A.1 |
| $\|S_C\|$, $\|S_N\|$ | 268 concept + 320 neutral sentences, at most 40 per language × 8 (`experiments/multilingual_hp/data/`); the paper uses 300 English | §4.1 |
| $\tau$ (mass-ratio threshold) | 2.0 | App. A.3 |
| judge confidence | ≥ 0.85, with 20 tokens per feature | App. A.3 |
| $\delta$ (edit strength) | tuned per concept over {0.5, 1, 2, 5, 10, 50, 100, 200} | App. C.3 |

In [ ]:
import gc
import json
import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()  # HF_TOKEN / GEMINI_API_KEY from .env, if present
pd.set_option("display.max_colwidth", 200)

# ---- what to erase ------------------------------------------------------
CONCEPT = 'Gambling'
TAG = 'GAMBLING'            # short concept label in printouts
sys.path.insert(0, "experiments/multilingual_hp")
from splitting import LANGS, concept_paths, concept_slug, neutral_paths
NEUTRAL_SOURCE = "translated"   # only translated neutral sets exist for this concept
NEUTRAL_REP = 0                 # 0, 1 or 2: shared neutral replicate
SMOKE = False                   # True: quick end-to-end pass (short fit, 3 judged features, tiny evals)
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"     # gated: needs HF_TOKEN with access
SEED = 42

# ---- Step 3.1: Sparse MF (App. A.1) -------------------------------------
RANK = 200 if "llama" in MODEL_NAME.lower() else 100   # k; kept at the paper value though V* is larger with 8 languages
SPARSITY = 0.01          # s
RIDGE = 1e-4             # lambda
MAX_ITER = 20_000        # T
PATIENCE = 50

# ---- Step 3.2: feature selection (App. A.3) -----------------------------
TAU = 2.0
CONFIDENCE = 0.85
TOP_K_TOKENS = 20
GEMINI_MODEL = "models/gemini-3.5-flash-lite"   # repo default; the paper used Gemini-2.5-Flash-Lite

# ---- Step 3.3: erasure (App. C.3) ---------------------------------------
DELTA_GRID = [0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0, 200.0]
DELTA = 10.0             # Gemma value, not tuned for Llama; on Llama efficacy saturates around 50-100 (Fig. 11), see 3.4

# ---- overrides from scripts/run_overnight.py --param NAME=VALUE (set in an injected first cell) ----
globals().update(globals().get("_OVERRIDES", {}))
assert NEUTRAL_SOURCE == "translated", "only translated neutral sets exist for this concept"
assert NEUTRAL_REP in (0, 1, 2), NEUTRAL_REP
if SMOKE:
    MAX_ITER, PATIENCE = 50, 5

# ---- run directory: resume caches (smf.pt, judge.json, mc.json, gen.json) ----
RUN_DIR = Path("runs") / concept_slug(CONCEPT) / f"{NEUTRAL_SOURCE}_rep{NEUTRAL_REP}{'_smoke' if SMOKE else ''}"
RUN_DIR.mkdir(parents=True, exist_ok=True)


def load_json(path: Path, default):
    return json.loads(path.read_text()) if path.exists() else default


def save_json(path: Path, obj) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")   # write then rename, so a crash never leaves a half file
    tmp.write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
    tmp.replace(path)


concepts = [e["concept"] for e in json.loads(Path("data/concept_sentences.json").read_text())]
assert CONCEPT in concepts, f"{CONCEPT!r} is not one of: {concepts}"
assert os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN"), "Llama-3.1 is gated: set HF_TOKEN (e.g. in .env)"
print(f"concept={CONCEPT!r} | model={MODEL_NAME} | k={RANK} | languages={list(LANGS)} | "
      f"neutral={NEUTRAL_SOURCE} rep {NEUTRAL_REP} | smoke={SMOKE} | run dir={RUN_DIR}")


### 0.2 Gemini API key

Step 3.2 is the only step that calls Gemini. You can paste your key below or put `GEMINI_API_KEY=...`
in `.env`. Clear the key from this cell before you commit or share the notebook.

In [ ]:
GEMINI_API_KEY = ""  # paste here, or leave empty to use .env / the environment

if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
assert any(os.getenv(k) for k in ("GOOGLE_API_KEY", "GEMINI_API_TOKEN", "GEMINI_API_KEY")), \
    "No Gemini key found: Step 3.2 needs one."
print("Gemini key found.")

### 0.3 Repo imports

These are the building blocks the paper's pipeline uses. Reusing them means the notebook gives the same
results as `ember.train_mf_features`, `ember.interpret_features`, and `ember.run_erasure`.

In [ ]:
from llm_utils.activation_generator import ActivationGenerator, extract_token_ids_sample_ids_and_labels
from transformers import AutoTokenizer

from ember.erasure.model_loader import load_hf_model
from ember.evals.mc import evaluate_mc_generation, prepare_mc_items
from ember.evals.model_wrap import WrappedHFModel
from ember.interpret_features import (
    MEMBERSHIP_PROMPT,
    TOKENS_DESC_SYS,
    SimpleGeminiClient,
    classify_membership_with_desc,
    get_description,
)
from ember.local_datasets import ConceptDataset, load_mc_qa_items, load_open_qa_examples
from ember.timing import Timer
from ember.utils import (
    SparseMatrixFactorization,
    _safe_model_name,
    build_token_label_codes,
    collect_feature_rows_for_embeddings,
    compute_embedding_stats,
    fit_with_ridge,
    get_embedding_matrix,
    get_special_token_ids,
    resolve_device,
    set_seed,
)

set_seed(SEED)
# DEVICE = resolve_device("auto")
# This host's vGPU reports CUDA as available but prohibits compute, so force CPU.
DEVICE = "cpu"
os.environ["EMBER_DEVICE"] = "cpu"  # load_hf_model reads this
print(f"device: {DEVICE}")

---
# Step 3.1: Finding embedding features with Sparse Matrix Factorization

**Goal:** split each token embedding $\mathbf e_t$ into a sum of a few shared directions, so that
semantically related tokens (e.g. *casino*, *bet*, *poker*) share a feature:

$$E_{V^*}^\top \approx Z\,Y,\qquad Z\in\mathbb R^{d\times k}\ \text{(dense features)},\quad Y\in\mathbb R^{k\times|V^*|}\ \text{(WTA-sparse coefficients)}$$

The hard winner-takes-all (WTA) constraint on the rows of $Y$ means each feature $\mathbf z_i$ is used by only
1% of the tokens in $V^*$. That is why a feature can be specific to one concept.

### 3.1.a Load the model (TransformerLens)

The pipeline reads $E$ through TransformerLens. Unlike Gemma, Llama has no $\sqrt{d}$ embedding multiplier,
and TL does not center `W_E` for RMSNorm models, so TL's `W_E` is exactly HF's `embed_tokens` and Step 3.3
writes back with scale 1.

**Memory.** `ActivationGenerator` loads TL in fp32 (~32 GB for 8B, after first loading the HF weights), which does
not fit next to the later HF copies. The subclass below loads it in bf16, the dtype the HF checkpoint is stored in,
so $E$ is the same values the HF model uses.

In [ ]:
from transformer_lens import HookedTransformer


class BF16ActivationGenerator(ActivationGenerator):
    """ActivationGenerator whose TL model is loaded in bf16 instead of fp32."""

    def __init__(self, model_name, model_device="cpu", data_device="cpu", mode="mlp"):
        self.model = HookedTransformer.from_pretrained(model_name, device=model_device, dtype=torch.bfloat16)
        self.model_name = model_name
        self.data_device = data_device
        self._mode = mode


act_generator = BF16ActivationGenerator(MODEL_NAME, model_device=DEVICE, data_device="cpu", mode="mlp")
tl_model = act_generator.model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

E = get_embedding_matrix(tl_model)  # (|V|, d)
VOCAB_SIZE, D_MODEL = E.shape
print(f"E: {tuple(E.shape)}  dtype={E.dtype}")

### 3.1.b Build $S_C \cup S_N$ and the token set $V^*$

- $S_C$: up to 40 Wikipedia Gambling sentences per language (en 40, es 40, fr 40, de 40, it 40, pt 32, hi 26, th 10; 268 in total).
  $S_N$: 40 translated neutral sentences per language from replicate `NEUTRAL_REP` (320). A language without
  concept sentences shows 0 in the table.
- $V^*$: the unique non-special token ids in $S_C \cup S_N$. The model factorizes only these rows of $E$,
  which is cheaper and keeps the factorization focused on the concept. The neutral tokens let Step 3.2
  tell concept features apart from general ones.
- `sent_lang` maps every sentence to its language, and `tok_langs[t]` is the set of languages token $t$ occurs in.
  The table counts sentences per language and $V^*$ tokens per language (a token counts for every language it occurs in).

In [ ]:
CONCEPT_PATH = concept_paths(CONCEPT)["concept"]
NEUTRAL_PATH = neutral_paths(CONCEPT, NEUTRAL_REP)["base"]
dataset = ConceptDataset(concept_name=CONCEPT, concept_path=CONCEPT_PATH, neutral_path=NEUTRAL_PATH,
                         neutral_sample_seed=42)
token_ids_all, sample_ids, labels_all = extract_token_ids_sample_ids_and_labels(dataset, act_generator)

special = get_special_token_ids(tl_model)
vstar_ids = sorted({int(t) for t in token_ids_all.tolist()} - special)

# sentence -> language code ("langs" is aligned with "sentences"; neutral rows carry "lang")
c_entry = next(e for e in json.loads(CONCEPT_PATH.read_text()) if e["concept"] == CONCEPT)
sent_lang = dict(zip(c_entry["sentences"], c_entry["langs"]))
sent_lang.update({o["sentence"]: o["lang"] for o in json.loads(NEUTRAL_PATH.read_text())})
assert all(s in sent_lang for s, _ in dataset), "some dataset sentences have no language"

# token id -> set of languages it occurs in
tok_langs = {}
for s, _ in dataset:
    for t in tokenizer(s, add_special_tokens=False)["input_ids"]:
        tok_langs.setdefault(t, set()).add(sent_lang[s])

per_lang = pd.DataFrame(0, index=list(LANGS), columns=["|S_C|", "|S_N|", "V* tokens"])
for s, lbl in dataset:
    per_lang.loc[sent_lang[s], "|S_C|" if lbl == CONCEPT else "|S_N|"] += 1
for t in vstar_ids:
    for lang in tok_langs.get(t, ()):
        per_lang.loc[lang, "V* tokens"] += 1   # a token counts for every language it occurs in
per_lang.insert(0, "language", [LANGS[l][0] for l in per_lang.index])

n_concept_sents = sum(lbl == CONCEPT for _, lbl in dataset)
few = {l: int(n) for l, n in per_lang["|S_C|"].items() if n < 40}
if few:
    print(f"languages with fewer than 40 concept sentences: {few}")
print(f"|S_C| = {n_concept_sents}   |S_N| = {len(dataset) - n_concept_sents}   neutral source = {NEUTRAL_SOURCE} rep {NEUTRAL_REP}")
print(f"token occurrences = {len(token_ids_all)}  ->  |V*| = {len(vstar_ids)} "
      f"({len(vstar_ids) / VOCAB_SIZE:.2%} of the vocabulary)")
per_lang

### 3.1.c Build the data matrix $A = E_{V^*}^\top$

Each column of $A$ is one token's embedding.

In [ ]:
E_vstar = E.index_select(0, torch.tensor(vstar_ids, device=E.device)).detach().float()  # (|V*|, d)
A = E_vstar.T.contiguous().to(DEVICE)                                                      # (d, |V*|)

print(f"A: {tuple(A.shape)}  (d x |V*|)")
print(f"WTA keeps ceil(s*|V*|) = {math.ceil(SPARSITY * len(vstar_ids))} tokens per feature")

### 3.1.d Fit Sparse MF (alternating least squares + WTA)

The fit alternates two closed-form ridge updates:

$$Z \leftarrow A Y^\top (Y Y^\top + \lambda I)^{-1},\qquad Y \leftarrow (Z^\top Z + \lambda I)^{-1} Z^\top A$$

After each $Y$ update it applies WTA, which keeps the top-$s$ fraction of $|Y_{i,:}|$ and zeroes the rest.
It also rescales between the two factors. Initialization is $\mathcal N(0,1)$, and the fit stops early after
500 iterations without an improvement of more than $10^{-4}$. If the matrix inverse is singular,
`fit_with_ridge` retries with $\lambda \times 10$.

This is the slowest cell in Step 3.1. It prints progress every 50 iterations. The fit is saved to `RUN_DIR/smf.pt`
and loaded from there on a re-run.

In [ ]:
# Resume: a saved fit in RUN_DIR/smf.pt is loaded instead of re-fitting (delete it, and judge.json, to re-fit).
SMF_PATH = RUN_DIR / "smf.pt"
if SMF_PATH.exists():
    ck = torch.load(SMF_PATH)
    assert ck["vstar_ids"] == vstar_ids, f"{SMF_PATH} was fitted on a different V*: delete it to re-fit"
    Z, Y, ridge_used, t_fit = ck["Z"], ck["Y"], ck["ridge_used"], {"elapsed": ck["elapsed"]}
    print(f"loaded {SMF_PATH}")
else:
    set_seed(SEED)
    smf = SparseMatrixFactorization(RANK, fitting_device=DEVICE, g_sparsity=SPARSITY)
    with Timer() as t_fit:
        ridge_used = fit_with_ridge(smf, A, MAX_ITER, patience=PATIENCE, base_reg=RIDGE)

    Z = smf.F_.detach().float().cpu()      # (d, k)      embedding features z_i (columns)
    Y = smf.G_.detach().float().cpu().T    # (k, |V*|)   sparse coefficients
    torch.save({"Z": Z, "Y": Y, "ridge_used": ridge_used, "elapsed": t_fit["elapsed"], "vstar_ids": vstar_ids}, SMF_PATH)
print(f"\nfitted in {t_fit['elapsed']:.1f}s (lambda={ridge_used:g})  Z: {tuple(Z.shape)}  Y: {tuple(Y.shape)}")

### 3.1.e Check the factorization

Reconstruction quality and sparsity. Each feature should use ≈ $\lceil s|V^*|\rceil$ tokens. Tokens that
no feature covers are left as pure reconstruction error $\boldsymbol\varepsilon_t$.

In [ ]:
A_cpu = A.cpu()
rel_err = (torch.norm(A_cpu - Z @ Y) / torch.norm(A_cpu)).item()
nz = Y.abs() > 0
tokens_per_feature = nz.sum(1)
features_per_token = nz.sum(0)

print(f"relative reconstruction error ||A - ZY||_F / ||A||_F = {rel_err:.4f}")
print(f"tokens per feature : min={tokens_per_feature.min().item()}  max={tokens_per_feature.max().item()}")
print(f"features per token : mean={features_per_token.float().mean():.2f}  max={features_per_token.max().item()}  "
      f"uncovered tokens={int((features_per_token == 0).sum())}")

In [ ]:
print(Z.size())
print(Y.size())
print(RANK)

### 3.1.f Baseline: truncated SVD with the same $k$

The rank-$k$ truncated SVD $A_k = U_k \Sigma_k V_k^\top$ is the best possible rank-$k$ approximation
(Eckart–Young theorem), so no rank-$k$ factorization can have a lower error, including Sparse MF.
The gap between the two errors is what the WTA sparsity costs in reconstruction.

In [ ]:
U, S, Vh = torch.linalg.svd(A_cpu, full_matrices=False)
A_svd = U[:, :RANK] @ torch.diag(S[:RANK]) @ Vh[:RANK]
svd_rel_err = (torch.norm(A_cpu - A_svd) / torch.norm(A_cpu)).item()

svd_nnz = U[:, :RANK].numel() + RANK + Vh[:RANK].numel()
smf_nnz = Z.numel() + int(nz.sum())
print(f"rank-{RANK} truncated SVD : relative error = {svd_rel_err:.4f}   (parameters: {svd_nnz:,})")
print(f"rank-{RANK} Sparse MF     : relative error = {rel_err:.4f}   (nonzero parameters: {smf_nnz:,})")

# Smallest SVD rank whose error is no worse than Sparse MF's.
tail = torch.sqrt(torch.flip(torch.cumsum(torch.flip(S ** 2, [0]), 0), [0]) / (S ** 2).sum())
matching_rank = int((tail <= rel_err).nonzero()[0]) if (tail <= rel_err).any() else len(S)
print(f"SVD reaches the Sparse MF error ({rel_err:.4f}) at rank {matching_rank}")

---
# Step 3.2: Identifying concept-related features

Most of the $k$ features are general (punctuation, function words, and so on). The selection runs in two filters:

1. **Mass ratio** $\rho_i$: is feature $i$ used mostly by concept tokens rather than neutral tokens? Keep the feature if $\rho_i \ge \tau = 2.0$.
2. **LLM judge** (Gemini, two stages): describe the feature's tokens *without* seeing the concept name,
   then ask whether that description is the concept. Keep the feature if `is_member` is true and confidence ≥ 0.85.

The features that pass both filters form $F_C$.

### 3.2.a Label every token in $V^*$

$\ell_t \in$ {concept, neutral, both}, depending on whether $t$ appears in $S_C$ only, $S_N$ only, or both.
Codes: `1` = concept, `0` = neutral, `2` = both.

In [ ]:
token_to_code = build_token_label_codes(CONCEPT, dataset, tokenizer)
codes = np.array([token_to_code.get(t, 0) for t in vstar_ids])

print(f"T_C (concept-only) = {(codes == 1).sum()}   T_N (neutral-only) = {(codes == 0).sum()}   "
      f"both = {(codes == 2).sum()}")

### 3.2.a′ Token → source word and language

Hindi and Thai Llama tokens are often byte pieces or sub-syllables (e.g. `ैर`, `อต`), which mean nothing on their own.
So for every token the cell records the word it most often came from, with that word's language:

- `tok_word[t] = (word, lang)`: the most frequent whole word containing the token's character span
  (whitespace-delimited; for Thai, which has no spaces, the pythainlp words the span overlaps);
- `tok_str[t]`: a readable form of the token (`decode`, or the raw BPE string when decoding gives only `�`).

The judge in 3.2.c–d sees `token → word (lang)`.

In [ ]:
import re
from collections import Counter

from pythainlp.tokenize import word_tokenize as th_word_tokenize

PUNCT = ".,;:!?()[]{}\"'«»“”‘’„–—।॥"


def word_spans(s: str, lang: str) -> list:
    """(start, end) of each word in s: whitespace-delimited, or pythainlp words for Thai."""
    if lang != "th":
        return [m.span() for m in re.finditer(r"\S+", s)]
    spans, pos = [], 0
    for w in th_word_tokenize(s, keep_whitespace=True):
        start = s.find(w, pos)
        if start < 0 or not w.strip():
            continue
        spans.append((start, start + len(w)))
        pos = start + len(w)
    return spans


word_counts = {}  # tid -> Counter of (word, lang)
for s, _ in tqdm(dataset, desc="token -> source word"):
    lang = sent_lang[s]
    enc = tokenizer(s, add_special_tokens=False, return_offsets_mapping=True)
    spans = word_spans(s, lang)
    for tid, (a, b) in zip(enc["input_ids"], enc["offset_mapping"]):
        a += len(s[a:b]) - len(s[a:b].lstrip())   # Llama folds the leading space into the token
        if a >= b:
            continue
        words = [s[ws:we] for ws, we in spans if ws < b and we > a]
        word = ("" if lang == "th" else " ").join(words).strip(PUNCT) or s[a:b]
        word_counts.setdefault(tid, Counter())[(word, lang)] += 1

tok_word = {t: c.most_common(1)[0][0] for t, c in word_counts.items()}


def readable(tid: int) -> str:
    s = tokenizer.decode([tid]).strip()
    return s if s.strip("�") else tokenizer.convert_ids_to_tokens(tid)


tok_str = {t: readable(t) for t in vstar_ids}
missing = [t for t in vstar_ids if t not in tok_word]
print(f"{len(tok_word)} tokens mapped to a source word; {len(missing)} V* tokens without one")

# a few examples per language
pd.DataFrame([
    {"lang": lang, "token": tok_str[t], "word": tok_word[t][0]}
    for lang in LANGS
    for t in [t for t in vstar_ids if t in tok_word and tok_word[t][1] == lang][:5]
])

### 3.2.b Score features by mass ratio and keep $\rho_i \ge \tau$

As implemented in `compute_embedding_stats` (column `ratio_abs`):

$$\rho_i = \frac{\operatorname{mean}_{t\in T_C} |Y_{i,t}|}{\operatorname{mean}_{t\in T_N} |Y_{i,t}|}$$

Tokens labeled *both* are excluded from the ratio. The threshold sweep below is the per-concept view of
Figure 6 in the paper.

In [ ]:
stats = compute_embedding_stats(Y.T, vstar_ids, token_to_code, CONCEPT).sort_values("ratio_abs", ascending=False)

for thr in [1.0, 1.25, 1.5, 1.75, 2.0, 2.25]:
    print(f"rho >= {thr:<4}: {(stats['ratio_abs'] >= thr).sum():3d} / {RANK} features")

candidates = stats[stats["ratio_abs"] >= TAU]
print(f"\n{len(candidates)} candidate features pass tau = {TAU}")
candidates[["feature", "ratio_abs", "mean_abs_concept", "mean_abs_neutral", "num_concept", "num_neutral", "num_both"]]

### 3.2.c Collect the tokens the judge will see

For each candidate, the code takes the tokens with nonzero $Y_{i,t}$, sorted by $|Y_{i,t}|$, and keeps the top 20.
The paper text says "concept-labeled tokens", but the released code sends the top activating tokens
whatever their label. This notebook follows the code. Each judge token is shown as `token → word (lang)`
(from 3.2.a′), built straight from $Y$ and `vstar_ids`. `feat_tokens` (plain token strings and labels) is
still built for the inspection cells below.

In [ ]:
token_label_map = {tid: "both" if c == 2 else CONCEPT if c == 1 else "Neutral" for tid, c in token_to_code.items()}
feature_rows = collect_feature_rows_for_embeddings(
    Y.T, vstar_ids, token_label_map, tl_model, _safe_model_name(MODEL_NAME), CONCEPT, RANK,
)
feat_tokens = pd.DataFrame(feature_rows).set_index("feature")[["activating_tokens", "labels"]]



def ml_judge_tokens(f: int, n: int = TOP_K_TOKENS) -> list:
    """Top-n support tokens of feature f by |Y|, each as 'token → word (lang)'."""
    mags = Y[f].abs()
    top = torch.argsort(mags, descending=True)[:n]
    top = top[mags[top] > 0].tolist()
    out = []
    for j in top:
        tid = vstar_ids[j]
        word, lang = tok_word.get(tid, (tok_str[tid], "?"))
        out.append(f"{tok_str[tid]} → {word} ({lang})")
    return out


cand = candidates[["feature", "ratio_abs"]].join(feat_tokens, on="feature").reset_index(drop=True)
if SMOKE:
    cand = cand.head(3)  # smoke: judge only the 3 highest-rho candidates
cand["judge_tokens"] = cand["feature"].astype(int).apply(ml_judge_tokens)
cand[["feature", "ratio_abs", "judge_tokens"]]

### 3.2.d-0 Rate-limited Gemini client

The free tier allows **15 requests per minute per model**. Stage 1 and Stage 2 make one request per
candidate each, so 2 × (number of candidates) requests in total. This wrapper does three things:

- spaces requests at least `60 / RPM` seconds apart (with a 10% margin);
- on a per-minute 429, waits the `retry in Ns` delay the server reports, then retries;
- fails immediately on a *daily* quota error, because waiting would not help.

Answers are cached in `descriptions` and `verdicts` and saved to `RUN_DIR/judge.json` after every call. Re-running a stage only queries the features that
are still missing, so a crash never wastes quota you already spent.

In [ ]:
import re
import time

GEMINI_RPM = 15


class ThrottledGemini:
    def __init__(self, inner: SimpleGeminiClient, rpm: int = GEMINI_RPM, max_attempts: int = 6):
        inner.max_retries = 1  # retries are handled here, with the server's retry delay
        self.inner = inner
        self.interval = 60.0 / rpm * 1.1
        self.max_attempts = max_attempts
        self._last = 0.0

    def generate(self, prompt: str) -> str:
        for attempt in range(self.max_attempts):
            time.sleep(max(0.0, self._last + self.interval - time.monotonic()))
            self._last = time.monotonic()
            try:
                return self.inner.generate(prompt)
            except RuntimeError as e:
                msg = str(e)
                if "429" not in msg or "PerDay" in msg or attempt == self.max_attempts - 1:
                    raise
                m = re.search(r"retry in ([\d.]+)s", msg)
                time.sleep(float(m.group(1)) + 1 if m else 60)


USE_THROTTLE = False  # billing enabled; set True to go back to free-tier pacing

if USE_THROTTLE:
    client = ThrottledGemini(SimpleGeminiClient(model_name=GEMINI_MODEL))
else:
    client = SimpleGeminiClient(model_name=GEMINI_MODEL)

# Persisted in RUN_DIR/judge.json, saved after every Gemini call, so a restart re-queries nothing.
# Delete judge.json together with smf.pt when re-fitting Step 3.1.
JUDGE_PATH = RUN_DIR / "judge.json"
_judge = load_json(JUDGE_PATH, {})
descriptions = {int(f): d for f, d in _judge.get("descriptions", {}).items()}
verdicts = {int(f): tuple(v) for f, v in _judge.get("verdicts", {}).items()}


def save_judge() -> None:
    save_json(JUDGE_PATH, {"descriptions": descriptions, "verdicts": verdicts})


print(f"{2 * len(cand)} requests needed, {len(descriptions)} descriptions and {len(verdicts)} verdicts cached; "
      f"throttle={'on' if USE_THROTTLE else 'off'}")

### 3.2.d LLM judge, stage 1: describe each feature *without* the concept name

Stage 1 never sees the concept name. This avoids confirmation bias, where tokens like *luck* could
look like "Gambling" just because the judge was told the concept.

The prompt is the repo's `TOKENS_DESC_SYS` adapted to the 8 languages (`ML_TOKENS_DESC_SYS`). `get_description`
reads the module global `ember.interpret_features.TOKENS_DESC_SYS` at call time, so the cell overrides it there.

In [ ]:
import ember.interpret_features as ifx

ML_TOKENS_DESC_SYS = """
You get tokens that represent a single feature vector (with some noise).
Each token is shown as: token → the word it came from (language code).
The tokens can come from any of these 8 languages: English, Spanish, French, German, Italian, Portuguese, Hindi, Thai.
Some tokens are word pieces rather than whole words.
Infer the single most specific, cohesive concept shared by the relevant tokens, across languages. Phrase it in English.
Return only one concise sentence. No preface, no list, no caveats.

Example:
Tokens: ['▁tomorrow → tomorrow (en)', '▁demain → demain (fr)', '▁mañana → mañana (es)', '▁heute → heute (de)']
Explanation of feature behavior: this vector is related to specific days relative to now (e.g., today/tomorrow).
"""
ifx.TOKENS_DESC_SYS = ML_TOKENS_DESC_SYS  # get_description reads the module global at call time

print("---- Stage 1 prompt ----")
print(ML_TOKENS_DESC_SYS.strip(), "\n\nTokens: {tokens}\nExplanation of feature behavior:")

todo = cand[~cand["feature"].astype(int).isin(descriptions)]
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Stage 1: describe"):
    descriptions[int(row["feature"])] = get_description(client, row["judge_tokens"])
    save_judge()
cand["description"] = cand["feature"].map(descriptions)
cand[["feature", "ratio_abs", "description"]]

In [ ]:
N_SHOW = 10
LABEL_TAG = {1: "C", 0: "N", 2: "B"}  # concept / neutral / both

for _, row in cand.sort_values("ratio_abs", ascending=False).iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini : {descriptions.get(feat, '<not described yet>')}")
    print("  tokens : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

### 3.2.e LLM judge, stage 2: is the description *the* concept?

Stage 2 sees only the concept name and the Stage 1 description. It returns `{"is_member", "confidence"}`.
A feature joins $F_C$ only if `is_member` is true **and** confidence ≥ 0.85.

In [ ]:
print("---- Stage 2 prompt ----")
print(MEMBERSHIP_PROMPT.format(concept=CONCEPT, desc="{description}").strip())

todo = {f: descriptions[f] for f in cand["feature"].astype(int).tolist() if f in descriptions and f not in verdicts}
for feat, desc in tqdm(todo.items(), desc="Stage 2: classify"):
    verdicts[feat] = classify_membership_with_desc(client, CONCEPT, desc)
    save_judge()

cand["is_member"] = cand["feature"].map(lambda f: verdicts[int(f)][0])
cand["confidence"] = pd.to_numeric(cand["feature"].map(lambda f: verdicts[int(f)][1]), errors="coerce").fillna(0.0)
cand["selected"] = (cand["is_member"] == True) & (cand["confidence"] >= CONFIDENCE)

F_C = sorted(cand.loc[cand["selected"], "feature"].astype(int).tolist())
print(f"\nF_C = {F_C}  ({len(F_C)} concept-related features)")
cand[["feature", "ratio_abs", "is_member", "confidence", "selected", "description"]]

In [ ]:
is_true = cand["is_member"] == True
ordered = cand.assign(
    _group=(~is_true).astype(int),
    _key=np.where(is_true, -cand["confidence"], cand["confidence"]),
).sort_values(["_group", "_key"])

for _, row in ordered.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    is_member, conf = verdicts.get(feat, (None, None))
    verdict = "<not classified yet>" if feat not in verdicts else (
        f"is_member={is_member}  confidence={conf if conf is not None else 'n/a'}  "
        f"-> {'SELECTED' if row['selected'] else 'rejected'}"
    )

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {descriptions.get(feat, '<not described yet>')}")
    print(f"  verdict : {verdict}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

In [ ]:
members = cand[cand["is_member"] == True].sort_values("ratio_abs", ascending=False)
print(f"{len(members)} / {len(cand)} candidates have is_member=True\n")

for _, row in members.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {row['description']}")
    print(f"  verdict : confidence={row['confidence']:.2f}  -> {'SELECTED' if row['selected'] else 'rejected (confidence < ' + str(CONFIDENCE) + ')'}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

### 3.2.f Inspect the selected features

Table 3 of the paper reports only 1–3 embedding features per concept.
Each selected feature should be a tight cluster of concept-specific tokens.

In [ ]:
if not F_C:
    print("No feature passed both filters: Step 3.3 will be a no-op for this concept.")

for feat in F_C:
    row = cand.loc[cand["feature"] == feat].iloc[0]
    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f}  conf={row['confidence']:.2f} ===")
    print(f"  {row['description']}")
    print("  tokens:", ", ".join(f"{t!r}[{l[0]}]" for t, l in zip(row["activating_tokens"][:30], row["labels"][:30])))
    print()

In [ ]:
# One column per selected feature: every token with nonzero Y_{i,t}, sorted by |Y_{i,t}|.
columns = {}
for feat in F_C:
    mags = Y[feat].abs()
    nz_idx = torch.nonzero(mags > 0).flatten()
    nz_idx = nz_idx[torch.argsort(mags[nz_idx], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in nz_idx])
    columns[f"feature {feat}"] = pd.Series(
        [f"{t} [{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, nz_idx)]
    )

wide = pd.DataFrame(columns).fillna("")
wide.index = range(1, len(wide) + 1)

all_cols = sorted({j for feat in F_C for j in torch.nonzero(Y[feat].abs() > 0).flatten().tolist()})
print(f"{len(F_C)} concept features, {len(all_cols)} distinct tokens "
      f"(C={sum(codes[j] == 1 for j in all_cols)}, N={sum(codes[j] == 0 for j in all_cols)}, "
      f"B={sum(codes[j] == 2 for j in all_cols)})")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(wide)

In [ ]:
TOP_N = 30
E_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).float().cpu()  # E_vstar is deleted in 3.3.a
token_strs = tokenizer.convert_ids_to_tokens(vstar_ids)

group_masks = {
    "C": codes == 1,
    "C / B": (codes == 1) | (codes == 2),
    "C / B / N": np.ones_like(codes, dtype=bool),
}

for feat in F_C:
    y = Y[feat]
    # z's sign is arbitrary (Y can absorb it); orient it so its support tokens mostly have positive Y.
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots = E_ref @ z
    cos = dots / (E_ref.norm(dim=1) * z.norm())

    df = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot": dots.numpy(),
        "cos": cos.numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df[mask].sort_values("dot", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t, z> --")
        display(top)

In [ ]:
E_unit = E_ref / E_ref.norm(dim=1, keepdim=True)

for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots_unit = E_unit @ z

    df_unit = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot_unit": dots_unit.numpy(),
        "||e_t||": E_ref.norm(dim=1).numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df_unit[mask].sort_values("dot_unit", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t / ||e_t||, z> --")
        display(top)

In [ ]:
TOP_K_COUNT = 100

rows = []
for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    rankings = {
        "<e_t, z>": E_ref @ z,
        "<e_t / ||e_t||, z>": E_unit @ z,
    }
    for name, scores in rankings.items():
        top = torch.argsort(scores, descending=True)[:TOP_K_COUNT].numpy()
        rows.append({
            "feature": feat,
            "ranking": name,
            "C": int((codes[top] == 1).sum()),
            "B": int((codes[top] == 2).sum()),
            "N": int((codes[top] == 0).sum()),
            "C in support": int(((codes[top] == 1) & (y[top] != 0).numpy()).sum()),
        })

print(f"Label counts among the top {TOP_K_COUNT} of all {len(vstar_ids)} tokens in V* "
      f"(base rate: C={(codes == 1).mean():.1%}, B={(codes == 2).mean():.1%}, N={(codes == 0).mean():.1%})")
pd.DataFrame(rows)

### 3.2.g Language mix of the selected features

For each $i \in F_C$: its support tokens ($Y_{i,t}\neq 0$) counted per language and label (C / N / B), and each
language's share of the feature's mass $\sum_t |Y_{i,t}|$ (a token that occurs in $k$ languages gives $1/k$ of
its mass to each). Mass spread over many languages means the Gambling direction is shared across languages; mass in
one or two languages (or one script) means it is language-specific, and erasing it may not reach the others.

In [ ]:
for feat in F_C:
    support = torch.nonzero(Y[feat] != 0).flatten().tolist()
    counts = pd.DataFrame(0, index=list(LANGS) + ["?"], columns=["C", "N", "B"])
    mass = pd.Series(0.0, index=counts.index)
    for j in support:
        langs = tok_langs.get(vstar_ids[j]) or {"?"}
        for lang in langs:
            counts.loc[lang, LABEL_TAG[codes[j]]] += 1   # a token counts for every language it occurs in
            mass[lang] += Y[feat, j].abs().item() / len(langs)

    mix_df = counts.assign(total=counts.sum(1), mass_share=mass / mass.sum())
    mix_df = mix_df[mix_df["total"] > 0]
    mix_df.insert(0, "language", [LANGS[l][0] if l in LANGS else "?" for l in mix_df.index])
    print(f"=== feature {feat}: {len(support)} support tokens ===")
    display(mix_df.style.format({"mass_share": "{:.1%}"}))

In [ ]:
# ---- Dense refit: fix Z, solve Y' = argmin ||E*^T - Z Y'||_F (no sparsity) ----
TOP_DIRS = 5
HP_FEATS = list(F_C)          # the concept column(s) of Z; override e.g. [37] if you want one
HP_MARK = f"★★★ {TAG} ★★★"

A_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).double().cpu().T   # (d, |V*|)
Z64 = Z.double()                                                                    # (d, k)
Y_dense = torch.linalg.lstsq(Z64, A_ref).solution.float()                           # (k, |V*|)

dense_err = (torch.norm(A_ref - Z64 @ Y_dense.double()) / torch.norm(A_ref)).item()
sparse_err = (torch.norm(A_ref - Z64 @ Y.double()) / torch.norm(A_ref)).item()
print(f"relative error: sparse Y = {sparse_err:.4f}   dense Y' = {dense_err:.4f}")
print(f"{TAG} feature(s): {HP_FEATS}   (||z|| = {[round(Z[:, f].norm().item(), 3) for f in HP_FEATS]})\n")

for hp in HP_FEATS:
    mags = Y[hp].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    print(f"{'=' * 100}\n=== {TAG} feature {hp}: {len(support)} tokens with nonzero Y, by decreasing |Y| ===\n{'=' * 100}")
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        top = order[:TOP_DIRS]

        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Y[{hp}] = {Y[hp, j]:+.3f}")
        for r, f in enumerate(top, 1):
            mark = f"   <-- {HP_MARK}" if f in HP_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")
        for f in HP_FEATS:
            if f not in top:
                print(f"       ..  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}   <-- {HP_MARK} (rank {order.index(f) + 1}/{len(order)})")

In [ ]:
# ---- Tokens where the Gambling direction dominates Y': top-1 is Gambling and |Y'_1| >= RATIO * |Y'_2| ----
RATIO = 2.0

for hp in HP_FEATS:
    mags = Y[hp].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    kept = []
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        first, second = y_dense_t[order[0]].abs().item(), y_dense_t[order[1]].abs().item()
        if order[0] == hp and first >= RATIO * second:
            kept.append((rank_t, j, tok, order, first / second))

    print(f"{'=' * 100}\n=== {TAG} feature {hp}: {len(kept)} / {len(support)} tokens where z_{hp} is top-1 "
          f"and |Y'| >= {RATIO:g}x the runner-up ===\n{'=' * 100}")
    for rank_t, j, tok, order, ratio in kept:
        y_dense_t = Y_dense[:, j]
        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Y[{hp}] = {Y[hp, j]:+.3f}   "
              f"top-1/top-2 = {ratio:.2f}x")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {HP_MARK}" if f in HP_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")

In [ ]:
# ---- Tokens skipped by the previous cell: Gambling is not top-1, or top-1 but < RATIO x the runner-up ----
for hp in HP_FEATS:
    mags = Y[hp].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    skipped = []
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        first, second = y_dense_t[order[0]].abs().item(), y_dense_t[order[1]].abs().item()
        if order[0] != hp:
            skipped.append((rank_t, j, tok, order, f"{TAG} is not top-1 (rank {order.index(hp) + 1}/{len(order)})"))
        elif first < RATIO * second:
            skipped.append((rank_t, j, tok, order, f"{TAG} is top-1 but only {first / second:.2f}x the runner-up"))

    n_not_top = sum(r.startswith(f"{TAG} is not") for *_, r in skipped)
    print(f"{'=' * 100}\n=== {TAG} feature {hp}: {len(skipped)} / {len(support)} tokens skipped "
          f"({n_not_top} {TAG} not top-1, {len(skipped) - n_not_top} top-1 but < {RATIO:g}x) ===\n{'=' * 100}")
    for rank_t, j, tok, order, reason in skipped:
        y_dense_t = Y_dense[:, j]
        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Y[{hp}] = {Y[hp, j]:+.3f}   -> {reason}")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {HP_MARK}" if f in HP_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")
        if hp not in order[:TOP_DIRS]:
            print(f"       ..  z_{hp:<4d} Y' = {y_dense_t[hp]:+.3f}   <-- {HP_MARK} (rank {order.index(hp) + 1}/{len(order)})")

In [ ]:
# ---- All C-labelled tokens, sorted by |Y'_hp| / max_{i != hp} |Y'_i| (= top-1/top-2 when Gambling is top-1) ----
for hp in HP_FEATS:
    c_cols = np.where(codes == 1)[0].tolist()
    rows = []
    for j in c_cols:
        y_dense_t = Y_dense[:, j]
        others_max = torch.cat([y_dense_t[:hp], y_dense_t[hp + 1:]]).abs().max().item()
        rows.append((y_dense_t[hp].abs().item() / others_max, j))
    rows.sort(reverse=True)

    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for _, j in rows])
    n_pass = sum(r >= RATIO for r, _ in rows)
    n_top1 = sum(r >= 1.0 for r, _ in rows)
    print(f"{'=' * 100}\n=== {TAG} feature {hp}: {len(rows)} C tokens | {TAG} top-1: {n_top1} | "
          f"{TAG} top-1 and >= {RATIO:g}x: {n_pass} ===\n{'=' * 100}")

    shown_ratio_line = shown_top1_line = False
    for rank_t, ((ratio, j), tok) in enumerate(zip(rows, toks), 1):
        if not shown_ratio_line and ratio < RATIO:
            print(f"\n{'-' * 40} below {RATIO:g}x {'-' * 40}")
            shown_ratio_line = True
        if not shown_top1_line and ratio < 1.0:
            print(f"\n{'-' * 36} {TAG} no longer top-1 {'-' * 36}")
            shown_top1_line = True

        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        in_support = f"in {TAG} support" if Y[hp, j] != 0 else f"not in {TAG} support"
        print(f"\n{rank_t:3d}. {tok!r} [C]   ratio = {ratio:.2f}x   sparse Y[{hp}] = {Y[hp, j]:+.3f}  ({in_support})")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {HP_MARK}" if f in HP_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")
        if hp not in order[:TOP_DIRS]:
            print(f"       ..  z_{hp:<4d} Y' = {y_dense_t[hp]:+.3f}   <-- {HP_MARK} (rank {order.index(hp) + 1}/{len(order)})")

In [ ]:
# ---- How orthogonal is each Gambling direction to the span of the other Z columns? ----
Z64 = Z.double()
k = Z64.shape[1]


def angle_to_span(f: int, exclude=()) -> dict:
    """Angle between z_f and span{z_i : i != f, i not in exclude}."""
    others = [i for i in range(k) if i != f and i not in exclude]
    z = Z64[:, f]
    coef = torch.linalg.lstsq(Z64[:, others], z.unsqueeze(1)).solution.squeeze(1)
    proj = Z64[:, others] @ coef
    resid = z - proj
    sin = (resid.norm() / z.norm()).item()                  # 1 = orthogonal, 0 = inside the span
    cos_single = F.cosine_similarity(Z64[:, others], z.unsqueeze(1), dim=0)
    best = int(cos_single.abs().argmax())
    return {
        "feature": f,
        "angle_deg": math.degrees(math.asin(min(sin, 1.0))),
        "resid/||z||": sin,
        "proj/||z||": (proj.norm() / z.norm()).item(),
        "max |cos| to one z_i": cos_single.abs().max().item(),
        "closest z_i": others[best],
        "top-3 lstsq coef (i: c)": ", ".join(
            f"{others[i]}: {coef[i]:+.2f}" for i in torch.argsort(coef.abs(), descending=True)[:3].tolist()
        ),
    }


print(f"d = {Z64.shape[0]}, k = {k}  ->  span of the other {k - 1} columns is a {k - 1}-dim subspace of R^{Z64.shape[0]}")
print("For reference, a random direction would sit at about "
      f"{math.degrees(math.acos(math.sqrt((k - 1) / Z64.shape[0]))):.1f}° from a random {k - 1}-dim subspace\n")

hp_rows = [angle_to_span(f) for f in HP_FEATS]
if len(HP_FEATS) > 1:  # also measure against non-Gambling directions only
    hp_rows += [{**angle_to_span(f, exclude=HP_FEATS), "feature": f"{f} (vs non-{TAG} only)"} for f in HP_FEATS]
if hp_rows:
    display(pd.DataFrame(hp_rows).set_index("feature"))

# Baseline: same measurement for every feature, to see whether Gambling is unusual
all_angles = pd.Series({f: angle_to_span(f)["angle_deg"] for f in range(k)}).sort_values()
print("\nangle to span of the others, all features:")
print(all_angles.describe().round(2).to_string())
for f in HP_FEATS:
    pct = (all_angles < all_angles[f]).mean()
    print(f"{TAG} feature {f}: {all_angles[f]:.2f}°  ->  more orthogonal than {pct:.0%} of features "
          f"(rank {all_angles.index.get_loc(f) + 1}/{k}, ascending)")

In [ ]:
# ---- The 5 directions closest to the Gambling direction: angle + Gemini description ----
HP = HP_FEATS[0] if HP_FEATS else int(stats.iloc[0]["feature"])  # empty F_C: the top-rho feature
N_NEIGHBORS = 5
N_TOKENS_SHOW = 15

z_hp = Z[:, HP]
cos_all = F.cosine_similarity(Z, z_hp.unsqueeze(1), dim=0)        # (k,)
cos_all[HP] = 0.0
neighbors = torch.argsort(cos_all.abs(), descending=True)[:N_NEIGHBORS].tolist()


def describe(f: int) -> str:
    if f not in descriptions:  # only tau-passing features were described in Step 3.2
        descriptions[f] = get_description(client, ml_judge_tokens(f))  # same multilingual judge tokens as 3.2.c
        save_judge()
    return descriptions[f]


def top_tokens(f: int) -> str:
    mags = Y[f].abs()
    top = torch.argsort(mags, descending=True)[:N_TOKENS_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])
    return ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}]" for t, j in zip(toks, top))


print(f"=== {TAG} feature {HP} ===\n  Gemini : {describe(HP)}\n  tokens : {top_tokens(HP)}\n")

rows = []
for r, f in enumerate(neighbors, 1):
    c = cos_all[f].item()
    angle = math.degrees(math.acos(min(abs(c), 1.0)))   # angle between the lines, 0°..90°
    shared = int(((Y[HP] != 0) & (Y[f] != 0)).sum())
    rows.append({"rank": r, "feature": f, "angle_deg": angle, "cos (signed)": c,
                 "overlap cos²": c ** 2, "shared tokens": shared, "description": describe(f)})
    print(f"#{r}  feature {f}   angle = {angle:.1f}°   cos = {c:+.3f}   (cos² = {c ** 2:.0%} of z_{HP} along it)   "
          f"shared support tokens = {shared}")
    print(f"  Gemini : {describe(f)}")
    print(f"  tokens : {top_tokens(f)}\n")

In [ ]:
# ---- The 5 directions most orthogonal to the Gambling direction ----
N_ORTHO = 5

others = torch.tensor([f for f in range(Z.shape[1]) if f != HP])
most_ortho = others[torch.argsort(cos_all[others].abs())[:N_ORTHO]].tolist()

rows = []
for r, f in enumerate(most_ortho, 1):
    c = cos_all[f].item()
    angle = math.degrees(math.acos(min(abs(c), 1.0)))
    shared = int(((Y[HP] != 0) & (Y[f] != 0)).sum())
    rows.append({"rank": r, "feature": f, "angle_deg": angle, "cos (signed)": c,
                 "shared tokens": shared, "description": describe(f)})
    print(f"#{r}  feature {f}   angle = {angle:.2f}°   cos = {c:+.4f}   shared support tokens = {shared}")
    print(f"  Gemini : {describe(f)}")
    print(f"  tokens : {top_tokens(f)}\n")

display(pd.DataFrame(rows).set_index("rank"))

### 3.2.h Extended vocabulary: fixed-Z refit on a larger token set

Keeps $Z$ from 3.1 fixed and refits only a sparse $Y'$ on a much larger, multilingual token set:

- $S_C^{ext}$: every sentence of the Gambling article in all 8 languages, `concept_ext.json`
  (801 sentences: en 111, es 88, fr 179, de 191, it 164, pt 32, hi 26, th 10);
- $S_N^{ext}$: per language, as many neutral sentences as that language's concept sentences, at most one per page:
  distinct English pages translated by Gemini (en stays English), from replicate `NEUTRAL_REP`
  (`neutral_ext_translated_rep<NEUTRAL_REP>.json`).

Both sets are built ahead of time under `experiments/multilingual_hp/data/concepts/<concept>/extended/`, so this cell
makes no network calls. With `SMOKE`, each side keeps only its first 30 sentences per language. The cells only create new `*_ext` names and change nothing that later cells use. They work before and after 3.3.a.

In [ ]:
# ---- Extended E*: keep Z from 3.1 fixed, refit only a sparse Y' on a larger multilingual token set ----
# S_C^ext = every sentence of the concept article in each language; S_N^ext = per language the same number
# of translated neutral sentences, at most one per page (replicate NEUTRAL_REP).
# Y' is the Y-step of SparseMatrixFactorization.fit with F_ = Z frozen:
#   Y' = WTA_s( (Z^T Z + lambda I)^{-1} Z^T A_ext ),  WTA keeping ceil(s |V*_ext|) tokens per feature.
from collections import Counter

from ember.utils import wta_cols

RUN_JOINT_REFIT = False   # True also refits Z and Y jointly on A_ext, to compare time and error

# 1) sentences (built ahead of time under experiments/multilingual_hp/data/concepts/<concept>/extended/)
CONCEPT_EXT_PATH = concept_paths(CONCEPT)["extended"] / "concept_ext.json"
NEUTRAL_EXT_PATH = neutral_paths(CONCEPT, NEUTRAL_REP)["ext"]
ext_concept_rows = json.loads(CONCEPT_EXT_PATH.read_text())
ext_neutral_rows = json.loads(NEUTRAL_EXT_PATH.read_text())


def cap_per_lang(rows: list, n: int) -> list:
    """The first n rows of each language, file order kept."""
    seen, out = Counter(), []
    for r in rows:
        seen[r["lang"]] += 1
        if seen[r["lang"]] <= n:
            out.append(r)
    return out


if SMOKE:  # first 30 sentences per language per side
    ext_concept_rows, ext_neutral_rows = cap_per_lang(ext_concept_rows, 30), cap_per_lang(ext_neutral_rows, 30)
ext_concept = [r["sentence"] for r in ext_concept_rows]
ext_neutral = [r["sentence"] for r in ext_neutral_rows]
ext_sent_lang = {r["sentence"]: r["lang"] for r in ext_concept_rows + ext_neutral_rows}

# 2) extended token set V*_ext, labelled like build_token_label_codes (1 = concept, 0 = neutral, 2 = both),
#    plus the languages each token occurs in
special_ids = {tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id}
ext_tok_langs = {}
def token_set(sents):
    ids = set()
    for s in sents:
        for t in tokenizer(s, add_special_tokens=False)["input_ids"]:
            ids.add(t)
            ext_tok_langs.setdefault(t, set()).add(ext_sent_lang[s])
    return ids - special_ids
c_ids, n_ids = token_set(ext_concept), token_set(ext_neutral)
vstar_ext = sorted(c_ids | n_ids)
codes_ext = np.array([2 if t in c_ids and t in n_ids else 1 if t in c_ids else 0 for t in vstar_ext])
new_tokens = set(vstar_ext) - set(vstar_ids)

ext_per_lang = pd.DataFrame(0, index=list(LANGS), columns=["|S_C^ext|", "|S_N^ext|", "V*_ext tokens", "new (*)"])
for r in ext_concept_rows:
    ext_per_lang.loc[r["lang"], "|S_C^ext|"] += 1
for r in ext_neutral_rows:
    ext_per_lang.loc[r["lang"], "|S_N^ext|"] += 1
for t in vstar_ext:
    for lang in ext_tok_langs[t]:
        ext_per_lang.loc[lang, "V*_ext tokens"] += 1   # a token counts for every language it occurs in
        ext_per_lang.loc[lang, "new (*)"] += t in new_tokens
ext_per_lang.insert(0, "language", [LANGS[l][0] for l in ext_per_lang.index])
print(f"S_C^ext = {len(ext_concept)}   S_N^ext = {len(ext_neutral)}   neutral source = {NEUTRAL_SOURCE} rep {NEUTRAL_REP}")
print(f"|V*_ext| = {len(vstar_ext)}  (|V*| = {len(vstar_ids)}, {len(new_tokens)} new, "
      f"{len(set(vstar_ids) - set(vstar_ext))} of V* not in V*_ext)")
display(ext_per_lang)

# 3) A_ext = E_{V*_ext}^T in the factorization space (TL rows; after 3.3.a, HF rows x EMB_SCALE match them)
ids = torch.tensor(vstar_ext)
if "E" in globals():
    E_ext = E.index_select(0, ids.to(E.device)).detach().float().cpu()
else:
    E_ext = W_orig.index_select(0, ids.to(W_orig.device)).detach().float().cpu() * EMB_SCALE
A_ext = E_ext.T.contiguous().to(DEVICE)                                  # (d, |V*_ext|)

# 4) Y-step with Z frozen (same ops as SparseMatrixFactorization.fit, incl. ridge and WTA)
Z_dev = Z.to(DEVICE)
with Timer() as t_ext:
    inv_Z = torch.linalg.inv(Z_dev.T @ Z_dev + torch.eye(Z_dev.shape[1], device=DEVICE) * ridge_used)
    G_ext = A_ext.T @ Z_dev @ inv_Z                                       # (|V*_ext|, k), the G_ layout
    wta_cols(G_ext, pct_keep=SPARSITY)
    if DEVICE != "cpu":
        torch.cuda.synchronize()
Y_ext = G_ext.T.float().cpu()                                             # (k, |V*_ext|)
A_ext_cpu = A_ext.cpu()
ext_err = (torch.norm(A_ext_cpu - Z @ Y_ext) / torch.norm(A_ext_cpu)).item()
print(f"\nfixed-Z fit: {t_ext['elapsed']:.3f}s (joint fit in 3.1.d: {t_fit['elapsed']:.1f}s)   "
      f"WTA keeps {math.ceil(SPARSITY * len(vstar_ext))} tokens per feature")
print(f"relative error on A_ext: fixed Z = {ext_err:.4f}   (3.1 joint fit on A: {rel_err:.4f})")

# 5) the selected concept features on the extended token set: support by language x label
for f in F_C:
    sup = torch.nonzero(Y_ext[f] != 0).flatten().tolist()
    old_sup = {vstar_ids[j] for j in torch.nonzero(Y[f]).flatten().tolist()}
    counts = pd.DataFrame(0, index=list(LANGS), columns=["C", "N", "B"])
    mass = pd.Series(0.0, index=counts.index)
    for j in sup:
        langs = ext_tok_langs[vstar_ext[j]]
        for lang in langs:
            counts.loc[lang, LABEL_TAG[codes_ext[j]]] += 1   # a token counts for every language it occurs in
            mass[lang] += Y_ext[f, j].abs().item() / len(langs)
    mix_ext = counts.assign(total=counts.sum(1), mass_share=mass / mass.sum())
    mix_ext.insert(0, "language", [LANGS[l][0] for l in mix_ext.index])
    print(f"\n=== feature {f}: {len(sup)} tokens in Y'_ext support | {sum(codes_ext[j] == 1 for j in sup)} concept-only | "
          f"{sum(vstar_ext[j] in old_sup for j in sup)}/{len(old_sup)} of its 3.1 support kept | "
          f"{sum(vstar_ext[j] in new_tokens for j in sup)} new tokens ===")
    display(mix_ext.style.format({"mass_share": "{:.1%}"}))

# 6) optional: joint Z, Y refit on A_ext with the 3.1 settings, for the speed / error comparison
if RUN_JOINT_REFIT:
    set_seed(SEED)
    smf_ext = SparseMatrixFactorization(RANK, fitting_device=DEVICE, g_sparsity=SPARSITY)
    with Timer() as t_joint:
        fit_with_ridge(smf_ext, A_ext, MAX_ITER, patience=PATIENCE, base_reg=RIDGE)
    joint_err = (torch.norm(A_ext_cpu - smf_ext.F_.detach().cpu() @ smf_ext.G_.detach().cpu().T)
                 / torch.norm(A_ext_cpu)).item()
    print(f"\njoint refit on A_ext: {t_joint['elapsed']:.1f}s, relative error {joint_err:.4f}   "
          f"vs fixed Z: {t_ext['elapsed']:.3f}s, {ext_err:.4f}")

In [ ]:
# Token counts per group: original V* (3.1) vs extended V*_ext, split into tokens already in V* and new ones (*),
# per language (a token counts for every language it occurs in)
lab_v = {t: LABEL_TAG[int(c)] for t, c in zip(vstar_ids, codes)}
lab_x = {t: LABEL_TAG[int(c)] for t, c in zip(vstar_ext, codes_ext)}
rows = []
for t, l in lab_v.items():
    rows += [{"lang": lang, "label": l, "set": "V* (original)"} for lang in tok_langs.get(t, {"?"})]
for t, l in lab_x.items():
    for lang in ext_tok_langs[t]:
        rows.append({"lang": lang, "label": l, "set": "V*_ext (all)"})
        rows.append({"lang": lang, "label": l, "set": "V*_ext, new (*)" if t in new_tokens else "V*_ext, also in V*"})
group_counts = (pd.DataFrame(rows).groupby(["lang", "label", "set"]).size().unstack("set", fill_value=0)
                .reindex(columns=["V* (original)", "V*_ext (all)", "V*_ext, also in V*", "V*_ext, new (*)"], fill_value=0)
                .reindex(pd.MultiIndex.from_product([list(LANGS), ["C", "N", "B"]], names=["lang", "label"]), fill_value=0))
group_counts["V*_ext / V*"] = (group_counts["V*_ext (all)"] / group_counts["V* (original)"].replace(0, np.nan)).round(2)
with pd.option_context("display.max_rows", None):
    display(group_counts)

# Tokens in both sets whose label changed (e.g. N in V* becomes B because the token also occurs in a new concept sentence)
both_sets = [t for t in vstar_ids if t in lab_x]
display(pd.crosstab(pd.Series([lab_v[t] for t in both_sets], name="label in V*"),
                    pd.Series([lab_x[t] for t in both_sets], name="label in V*_ext"), margins=True))

In [ ]:
# The selected concept features on the extended token set, ranked by |Y'| (signed value shown).
# Tokens shown as 'token → word (lang)', with the word looked up in the extended sentences.
from collections import Counter

ext_word_counts = {}  # tid -> Counter of (word, lang), over S_C^ext and S_N^ext (word_spans / PUNCT from 3.2.a′)
for s in tqdm(ext_concept + ext_neutral, desc="token -> source word (extended)"):
    lang = ext_sent_lang[s]
    enc = tokenizer(s, add_special_tokens=False, return_offsets_mapping=True)
    spans = word_spans(s, lang)
    for tid, (a, b) in zip(enc["input_ids"], enc["offset_mapping"]):
        a += len(s[a:b]) - len(s[a:b].lstrip())   # Llama folds the leading space into the token
        if a >= b:
            continue
        words = [s[ws:we] for ws, we in spans if ws < b and we > a]
        word = ("" if lang == "th" else " ").join(words).strip(PUNCT) or s[a:b]
        ext_word_counts.setdefault(tid, Counter())[(word, lang)] += 1
ext_tok_word = {t: c.most_common(1)[0][0] for t, c in ext_word_counts.items()}

for f in F_C:
    mags = Y_ext[f].abs()
    sup = torch.nonzero(mags > 0).flatten()
    sup = sup[torch.argsort(mags[sup], descending=True)].tolist()
    old_sup = {vstar_ids[j] for j in torch.nonzero(Y[f]).flatten().tolist()}
    print(f"=== feature {f}: {len(sup)} tokens in Y'_ext support | {sum(codes_ext[j] == 1 for j in sup)} concept-only | "
          f"{sum(vstar_ext[j] in old_sup for j in sup)}/{len(old_sup)} of its 3.1 support kept | "
          f"{sum(vstar_ext[j] in new_tokens for j in sup)} new tokens (*) ===")
    toks = [vstar_ext[j] for j in sup]
    table = pd.DataFrame({
        "token": [f"{readable(t)} → {ext_tok_word.get(t, ('?', '?'))[0]}" for t in toks],
        "lang": [ext_tok_word.get(t, ('?', '?'))[1] for t in toks],
        "label": [LABEL_TAG[codes_ext[j]] + ("*" if vstar_ext[j] in new_tokens else "") for j in sup],
        "all langs": [",".join(sorted(ext_tok_langs[t])) for t in toks],
        f"Y'[{f}]": [Y_ext[f, j].item() for j in sup],
        "in 3.1 support": [vstar_ext[j] in old_sup for j in sup],
    })
    table.index = pd.RangeIndex(1, len(table) + 1, name="rank")
    display(table.style.format({f"Y'[{f}]": "{:+.4f}"}))

Language composition of each concept direction: first the original fit (3.1), then the extended refit.

In [ ]:
# Same composition table for the ORIGINAL fit (3.1): support of Y[f] over V*, with tok_langs from 3.1.b
def lang_split_orig(cols, weights=None):
    out = pd.Series(0.0, index=list(LANGS))
    for i, j in enumerate(cols):
        langs = tok_langs.get(vstar_ids[j]) or set(LANGS)
        w = 1.0 if weights is None else float(weights[i])
        for lang in langs:
            out[lang] += w / len(langs)
    return out / out.sum()

base_orig = lang_split_orig(range(len(vstar_ids)))
for f in F_C:
    sup = torch.nonzero(Y[f] != 0).flatten().tolist()
    mags = Y[f, sup].abs().numpy()
    comp = pd.DataFrame({
        "language": [LANGS[l][0] for l in LANGS],
        "any": [sum(l in tok_langs.get(vstar_ids[j], set()) for j in sup) for l in LANGS],
        "only": [sum(tok_langs.get(vstar_ids[j]) == {l} for j in sup) for l in LANGS],
        "only, C": [sum(tok_langs.get(vstar_ids[j]) == {l} and codes[j] == 1 for j in sup) for l in LANGS],
        "share": lang_split_orig(sup).values,
        "mass": lang_split_orig(sup, mags).values,
        "base": base_orig.values,
    }, index=list(LANGS))
    comp["enrich"] = comp["mass"] / comp["base"]
    print(f"=== feature {f} (original fit): {len(sup)} support tokens ===")
    display(comp.style.format({"share": "{:.1%}", "mass": "{:.1%}", "base": "{:.1%}", "enrich": "{:.2f}"}))

In [ ]:
# Language composition of each extended concept direction: the tokens with Y'_ext[f] != 0 (the support), per language.
# A token can occur in several languages (e.g. a name spelled the same in en/es/fr/de/it/pt), so three views:
#   any      : number of support tokens that occur in this language (a token counts for each of its languages)
#   only     : support tokens that occur only in this language
#   share    : token count split 1/k over the token's k languages -> sums to 100%
#   mass     : same split, weighted by |Y'_ext[f, t]| -> where the direction's weight sits
#   base     : the share column for all of V*_ext -> what share you'd expect if the feature picked tokens at random
#   enrich   : mass / base  (> 1 = this language is over-represented in the direction)
def lang_split(cols, weights=None):
    out = pd.Series(0.0, index=list(LANGS))
    for i, j in enumerate(cols):
        langs = ext_tok_langs[vstar_ext[j]]
        w = 1.0 if weights is None else float(weights[i])
        for lang in langs:
            out[lang] += w / len(langs)
    return out / out.sum()

base = lang_split(range(len(vstar_ext)))
for f in F_C:
    sup = torch.nonzero(Y_ext[f] != 0).flatten().tolist()
    mags = Y_ext[f, sup].abs().numpy()
    comp = pd.DataFrame({
        "language": [LANGS[l][0] for l in LANGS],
        "any": [sum(l in ext_tok_langs[vstar_ext[j]] for j in sup) for l in LANGS],
        "only": [sum(ext_tok_langs[vstar_ext[j]] == {l} for j in sup) for l in LANGS],
        "only, C": [sum(ext_tok_langs[vstar_ext[j]] == {l} and codes_ext[j] == 1 for j in sup) for l in LANGS],
        "share": lang_split(sup).values,
        "mass": lang_split(sup, mags).values,
        "base": base.values,
    }, index=list(LANGS))
    comp["enrich"] = comp["mass"] / comp["base"]
    n_multi = sum(len(ext_tok_langs[vstar_ext[j]]) > 1 for j in sup)
    print(f"=== feature {f}: {len(sup)} support tokens, {n_multi} occur in more than one language ===")
    display(comp.style.format({"share": "{:.1%}", "mass": "{:.1%}", "base": "{:.1%}", "enrich": "{:.2f}"}))

---
# Step 3.3: Erasing concept-related features

The factorization writes each embedding as a sum of feature contributions plus residual error:

$$\mathbf e_t = \underbrace{\sum_{i\in F_C} Y_{i,t}\mathbf z_i}_{\mathbf c_t\ \text{(concept part)}} + \sum_{i\notin F_C} Y_{i,t}\mathbf z_i + \boldsymbol\varepsilon_t
\quad\Longrightarrow\quad \mathbf e_t' = \mathbf e_t - \delta\,\mathbf c_t$$

The edit applies **only** to concept-only tokens ($t \in T_C$) that have $Y_{i,t}\neq 0$ for some $i\in F_C$.
All other rows of $E$ stay unchanged, and so does the unembedding $U$. With $\delta=1$ the edit removes exactly
the recovered concept component. With $\delta>1$ it over-subtracts, which the paper finds often erases better.

This step keeps **two models** side by side: `original_model` (untouched) and `erased_model`
(the same weights with the EMBER edit applied), plus a small API to query and compare them.


### 3.3.a Swap the TL model for the original HF model

Erasure and evaluation run on HF models. Llama-3.1-8B's `lm_head` is already separate from the input
embeddings, so editing $E$ leaves $U$ unchanged, as the paper does. Then a sanity check confirms that
HF `W_E` × scale matches the TL rows the model factorized. That match is what makes the write-back in 3.3.c correct.

In [ ]:
# Re-run safe: the TL objects are freed on the first run, and original_model is never modified.
if "E_vstar" in globals():
    E_vstar_tl = E_vstar.cpu()  # factorization-space reference rows
    del act_generator, tl_model, E, E_vstar, A, A_cpu
    globals().pop("smf", None)  # absent when 3.1.d loaded smf.pt
    gc.collect()
    torch.cuda.empty_cache()

if "original_model" not in globals():
    original_model, hf_tok = load_hf_model(MODEL_NAME)
    original_model.eval()
    original_model.config.use_cache = True  # loader disables the KV cache for training; we only generate

W_orig = original_model.get_input_embeddings().weight
EMB_SCALE = math.sqrt(W_orig.shape[1]) if "gemma" in MODEL_NAME.lower() else 1.0

with torch.no_grad():
    hf_rows = W_orig[torch.tensor(vstar_ids, device=W_orig.device)].float().cpu() * EMB_SCALE
cos = F.cosine_similarity(hf_rows, E_vstar_tl, dim=1)
norm_ratio = hf_rows.norm(dim=1) / E_vstar_tl.norm(dim=1)
print(f"EMB_SCALE={EMB_SCALE:.3f} | cos(HF*scale, TL): min={cos.min():.4f} | norm ratio: "
      f"[{norm_ratio.min():.3f}, {norm_ratio.max():.3f}]")
assert cos.min() > 0.99 and (norm_ratio - 1).abs().max() < 0.02, "HF and TL embedding spaces do not match"


### 3.3.b Compute the edit: which tokens are edited, and by how much

For each eligible token $t$, the cell computes $\mathbf c_t = \sum_{i\in F_C} Y_{i,t}\mathbf z_i$ and the relative edit magnitude
$\mu_t = \|\mathbf c_t\| / \|\mathbf e_t\|$ (App. F.1). The paper reports that EMBER edits at most ~0.03% (Gemma)
or ~0.14% (Llama) of the vocabulary.

In [ ]:
Y_C = Y[F_C]          # (|F_C|, |V*|)
Z_C = Z[:, F_C]       # (d, |F_C|)

touched = (Y_C.abs() > 0).any(dim=0).numpy()
eligible = touched & (codes == 1)
edit_cols = torch.as_tensor(np.where(eligible)[0], dtype=torch.long)
edit_ids = [vstar_ids[j] for j in edit_cols.tolist()]

C = (Z_C @ Y_C[:, edit_cols]).T                        # (n_edit, d): c_t per edited token
mu = C.norm(dim=1) / E_vstar_tl[edit_cols].norm(dim=1)

edit_table = pd.DataFrame({
    "token": hf_tok.convert_ids_to_tokens(edit_ids),
    "token_id": edit_ids,
    "mu": mu.numpy(),
}).sort_values("mu", ascending=False, ignore_index=True)

print(f"tokens touched by F_C: {touched.sum()}  ->  edited (concept-only): {len(edit_ids)} "
      f"= {len(edit_ids) / VOCAB_SIZE:.4%} of the vocabulary")
skipped = [vstar_ids[j] for j in np.where(touched & (codes != 1))[0]]
print(f"skipped (labelled neutral/both): {hf_tok.convert_ids_to_tokens(skipped)}")
edit_table

### 3.3.c Build the erased model: $\mathbf e_t \leftarrow \mathbf e_t - \delta\,\mathbf c_t$

Builds a copy of the model that owns only its **input embedding matrix** (≈1 GB) and shares every other tensor
(layers, norms, `lm_head`) with `original_model`, then applies the edit to its embedding rows only; `original_model`
is never touched. The edit is computed in the factorization's scaled space: read `W_E × scale`, subtract,
then divide by `scale`.

`set_erasure_strength(delta)` always rewrites the edited rows from the original values, so you can change
$\delta$ without reloading (the sweep in 3.3.f does this). `set_erasure_strength(0.0)` makes the two models identical.

Memory: two full bf16 copies of Llama-3.1-8B would be 32 GB; sharing makes the second one ≈1 GB.


In [ ]:
import copy


def share_all_but_embeddings(base):
    """A copy of `base` that owns only its input-embedding matrix; every other tensor is shared with `base`."""
    emb = base.get_input_embeddings().weight
    shared = {id(t): t for t in [*base.parameters(), *base.buffers()] if t is not emb}
    return copy.deepcopy(base, memo=shared)


erased_model = share_all_but_embeddings(original_model)

W_erased = erased_model.get_input_embeddings().weight
edit_idx = torch.tensor(edit_ids, dtype=torch.long)
with torch.no_grad():
    original_rows = W_orig.index_select(0, edit_idx.to(W_orig.device)).cpu().clone()


def apply_ember_edit(W: torch.Tensor, delta: float) -> None:
    """Set the edited rows of embedding matrix W to e_t - delta * c_t (always from the original rows)."""
    with torch.no_grad():
        e = original_rows.to(W.device).float() * EMB_SCALE
        e_new = e - delta * C.to(device=W.device, dtype=torch.float32)
        W.index_copy_(0, edit_idx.to(W.device), (e_new / EMB_SCALE).to(W.dtype))


def set_erasure_strength(delta: float) -> None:
    global ERASED_DELTA
    apply_ember_edit(W_erased, delta)
    ERASED_DELTA = delta  # part of the gen/mc cache key of `erased`


set_erasure_strength(DELTA)

# Sanity check: only the edited embedding rows differ, and the unembedding is identical.
with torch.no_grad():
    changed = (W_orig.to(W_erased.device) != W_erased).any(dim=1).nonzero().flatten().tolist()
lm_head_same = torch.equal(original_model.get_output_embeddings().weight,
                           erased_model.get_output_embeddings().weight)
print(f"delta={DELTA}: {len(changed)} embedding rows differ ({len(edit_ids)} eligible); "
      f"lm_head identical: {lm_head_same}")
assert set(changed) <= set(edit_ids) and lm_head_same


### 3.3.d Query API: original vs. erased

| Function | What it does |
|---|---|
| `ask(question, model="original" or "erased")` | one greedy answer from one model (chat template applied; cached in `RUN_DIR/gen.json`) |
| `compare(questions)` | prints each question with both models' answers one under the other |
| `mc_accuracy(model)` | the paper's MC protocol on the test split: concept and similar-domain accuracy (cached in `RUN_DIR/mc.json`) |
| `compare_mc()` | `mc_accuracy` for both models in one table |


In [ ]:
import textwrap

MODELS = {
    "original": WrappedHFModel(original_model, hf_tok),
    "erased": WrappedHFModel(erased_model, hf_tok),
}

mc_sets = {
    "concept": prepare_mc_items(load_mc_qa_items("qa_test", CONCEPT), seed=SEED),
    "similar-domain": prepare_mc_items(load_mc_qa_items("simdom_test", CONCEPT), seed=SEED),
}
if SMOKE:
    mc_sets = {name: items[:2] for name, items in mc_sets.items()}
GEN_TOKENS = 10 if SMOKE else 60   # default max_new_tokens of ask() / compare()

# Resume caches: generations in gen.json, MC accuracies in mc.json (`tuned` changes with BEST_DELTA: never cached).
GEN_PATH, MC_PATH = RUN_DIR / "gen.json", RUN_DIR / "mc.json"
gen_cache, mc_cache = load_json(GEN_PATH, {}), load_json(MC_PATH, {})


def cache_key(model: str):
    """Cache name of a model; `erased` carries its current delta. None = do not cache."""
    if model == "tuned":
        return None
    return f"erased@{ERASED_DELTA:g}" if model == "erased" else model


def ask(question: str, model: str = "erased", max_new_tokens: int = None) -> str:
    max_new_tokens = max_new_tokens or GEN_TOKENS
    name = cache_key(model)
    key = json.dumps([name, question, max_new_tokens], ensure_ascii=False)
    if name is not None and key in gen_cache:
        return gen_cache[key]
    m = MODELS[model]
    out = m.generate(m.wrap_prompt(question), max_new_tokens=max_new_tokens, do_sample=False)
    if name is not None:
        gen_cache[key] = out
        save_json(GEN_PATH, gen_cache)
    return out


def compare(questions, max_new_tokens: int = None) -> None:
    for q in [questions] if isinstance(questions, str) else questions:
        print(f"Q: {q}")
        for name in MODELS:
            answer = textwrap.indent(ask(q, name, max_new_tokens), " " * 12).lstrip()
            print(f"  {name:<8}: {answer}")
        print()


def mc_accuracy(model: str) -> dict:
    name = cache_key(model)
    if name is not None and name in mc_cache:
        return mc_cache[name]
    out = {}
    for set_name, items in mc_sets.items():
        correct, invalid, _ = evaluate_mc_generation(MODELS[model], items)
        out[set_name] = correct / len(items)
        out[f"{set_name}_invalid"] = invalid
    if name is not None:
        mc_cache[name] = out
        save_json(MC_PATH, mc_cache)
    return out


def compare_mc() -> pd.DataFrame:
    return pd.DataFrame({name: mc_accuracy(name) for name in MODELS})

### 3.3.d′ Questions about Gambling in the 8 languages

`QUESTIONS_EN`: English questions about the concept. `QUESTIONS_ML`: one question per non-English language
(es, fr, de, it, pt, hi, th). `QUESTIONS_SIM`: similar-domain questions the erased model should still answer
(specificity). `compare()` asks both models every question.

Note: `compare_mc()` and the 3.4 $\delta$ tuning remain **English-only**, since only English QA data exists
(`qa_*` / `simdom_*` splits). They measure English efficacy and specificity only.

In [ ]:
# English questions about the concept, one per other language, and similar-domain questions (specificity)
QUESTIONS_EN = [
    "What is gambling?",
    "How does a casino make money?",
    "What is a slot machine?",
]
QUESTIONS_ML = [
    "¿Qué son los juegos de azar?",   # es
    "Qu'est-ce qu'un casino ?",   # fr
    "Was ist Glücksspiel?",   # de
    "Che cos'è il gioco d'azzardo?",   # it
    "O que é o jogo de azar?",   # pt
    "जुआ क्या है?",   # hi
    "การพนันคืออะไร",   # th
]
QUESTIONS_SIM = [
    "What is the stock market?",
    "What is a board game?",
]

In [ ]:
compare(QUESTIONS_EN + QUESTIONS_ML + QUESTIONS_SIM)

MC accuracy of `original` and `erased` on the test split (concept = efficacy, similar-domain = specificity).

In [ ]:
mc_test = compare_mc()
mc_test

### 3.3.g One erased model per $\delta$ in the grid
Builds an erased model for every $\delta$ in `DELTA_GRID` and registers each one in `MODELS` as `delta_<δ>`
(`delta_0.5`, `delta_1`, …, `delta_200`). You can then query any of them with the 3.3.d API:
`ask(QUESTIONS_EN[0], model="delta_50")`. With `SMOKE` the grid is [1, 10].

**Memory.** Eight full copies would not fit (about 16 GB each for Llama-3.1-8B).
The edit changes only rows of the input embedding matrix, so each copy owns its **own embedding matrix**
(about 1.05 GB: 128,256 × 4096 in bf16) and shares every other tensor (layers, norms, `lm_head`) with `original_model`.
Nothing ever writes to the shared tensors, so this is safe. The cell checks free RAM before it builds anything.

`compare()` and `compare_mc()` loop over all of `MODELS`, so after this cell they include the 8 extra models
too (slow on CPU). `compare_deltas()` prints `original` plus every `delta_<δ>` answer for a question.

In [ ]:
import psutil

if SMOKE:
    DELTA_GRID = [1.0, 10.0]

def delta_key(delta: float) -> str:
    return f"delta_{delta:g}"

todo = [d for d in DELTA_GRID if delta_key(d) not in MODELS]
emb_bytes = W_orig.numel() * W_orig.element_size()
full_bytes = sum(p.numel() * p.element_size() for p in original_model.parameters())
need, avail = len(todo) * emb_bytes, psutil.virtual_memory().available
print(f"full copy: {full_bytes / 2**30:.1f} GiB, so {len(DELTA_GRID)} full copies would need "
      f"{len(DELTA_GRID) * full_bytes / 2**30:.1f} GiB")
print(f"shared copy: {emb_bytes / 2**30:.2f} GiB each, so {len(todo)} to build need {need / 2**30:.1f} GiB "
      f"| available now: {avail / 2**30:.1f} GiB")
assert need < 0.8 * avail, "not enough free RAM for one embedding matrix per delta"

for delta in tqdm(todo, desc="building per-delta models"):
    m = share_all_but_embeddings(original_model)
    apply_ember_edit(m.get_input_embeddings().weight, delta)
    MODELS[delta_key(delta)] = WrappedHFModel(m, hf_tok)

# Sanity check: each copy owns only its embedding matrix and shares lm_head with the original.
orig_ptrs = {n: p.data_ptr() for n, p in original_model.named_parameters()}
for delta in DELTA_GRID:
    m = MODELS[delta_key(delta)].model
    own = [n for n, p in m.named_parameters() if p.data_ptr() != orig_ptrs[n]]
    assert own == ["model.embed_tokens.weight"], own
print(f"\nqueryable: {[delta_key(d) for d in DELTA_GRID]}   e.g. ask(q, model={delta_key(DELTA_GRID[-1])!r})")

def compare_deltas(questions, deltas=None, max_new_tokens: int = None) -> None:
    """Prints the original model's answer and every per-delta model's answer to each question."""
    names = ["original"] + [delta_key(d) for d in (deltas or DELTA_GRID)]
    for q in [questions] if isinstance(questions, str) else questions:
        print(f"Q: {q}")
        for name in names:
            answer = textwrap.indent(ask(q, name, max_new_tokens), " " * 14).lstrip()
            print(f"  {name:<10}: {answer}")
        print()

In [ ]:
for q in QUESTIONS_EN[:2] + QUESTIONS_ML[:2] + QUESTIONS_SIM[:1]:
    compare_deltas(q)

---
# 3.4 Tune $\delta$ with the paper's grid search (Stage 1, App. C.2)

The paper tunes EMBER's $\delta$ per concept in isolation ("Stage 1: Embedding grid"). This section
reproduces that protocol as implemented in `ember/erasure/methods/ember.py`, `ember/evals/orchestrator.py`
and `io.pick_best_embed_delta`:

| Item | Value |
|---|---|
| Grid | $\delta \in$ {0.5, 1, 2, 5, 10, 50, 100, 200} ($\delta=0$ is not evaluated) |
| Split | **validation** (`train` in the repo): 50 concept MC, 50 similar-domain MC, 50 MMLU, 150 AlpacaEval prompts |
| Normalization | MC and MMLU: $\text{clip}\big(\tfrac{\text{acc}-0.25}{\text{base}-0.25},0,1\big)$. Alpaca: $\min(1, \text{score}/\text{base})$. The base is the original model on the same split. |
| Axes | efficacy $= 1-\widehat{\text{qa}}$; specificity $= \text{HM}(\widehat{\text{mmlu}}, \widehat{\text{simdom}})$; coherence $= \text{HM}(\widehat{\text{instr}}, \widehat{\text{flu}})$ |
| Score | $H = \text{HM}(\text{efficacy}, \text{specificity}, \text{coherence})$ |
| Gates | $\widehat{\text{mmlu}} < 0.9 \Rightarrow H=0$ (the rest is skipped). $\widehat{\text{instr}}$ or $\widehat{\text{flu}} < 0.9 \Rightarrow H=0$. |
| Pick | largest $H$; ties go to the smaller $\delta$; if every $H=0$ then $\delta=0$ (no edit) |

The grid runs on a **third model copy, `tuned`**, so `original` and `erased` are untouched. At the end `tuned`
holds the best $\delta$ and is added to the query API.

**Cost.** AlpacaEval dominates: for each $\delta$ that passes the MMLU gate, 150 generations of up to
200 tokens plus 300 Gemini judge calls. This is slow on CPU. `TUNE_WITH_ALPACA = False` drops the coherence
axis, so $H = \text{HM}(\text{efficacy}, \text{specificity})$. That is faster but **not** the paper's protocol.
Memory: `tuned` shares everything but its embedding matrix with `original_model`, so it adds ≈1 GB.

Tuning uses the validation split; `compare_mc()` reports on the held-out test split, so there is no leakage.

**Multilingual note.** Only English QA / MMLU / AlpacaEval data exists, so efficacy and specificity here are English-only,
even though $F_C$ was found from 8 languages. Check the other languages by hand with the 3.3.d′ questions.

The §3.4 cells are tagged `skip-overnight`, so `scripts/run_overnight.py` does not run them.

In [ ]:
import threading

from ember.evals.alpaca import evaluate_alpaca
from ember.evals.gemini import GeminiEvaluator
from ember.evals.harmonic import harmonic_mean
from ember.evals.mmlu import evaluate_mmlu_generation, load_mmlu_items
from ember.local_datasets import load_alpaca_indices, load_mmlu_indices

TUNE_WITH_ALPACA = True   # paper protocol; False drops the coherence axis (faster, not the paper's H)
MIN_MMLU = 0.90           # repo EMBER gates (EMBERMethod.grid_eval_kwargs)
MIN_ALPACA = 0.90
ALPACA_BATCH = 8

val_mc = {
    "qa": prepare_mc_items(load_mc_qa_items("qa_train", CONCEPT), seed=SEED),
    "simdom": prepare_mc_items(load_mc_qa_items("simdom_train", CONCEPT), seed=SEED),
}
val_mmlu = load_mmlu_items(load_mmlu_indices("train"))
N_ALPACA = len(load_alpaca_indices("train"))
judge = GeminiEvaluator(model_name=GEMINI_MODEL) if TUNE_WITH_ALPACA else None


class ProgressModel:
    """Forwards to a WrappedHFModel and advances `bar` once per generated prompt."""

    def __init__(self, inner, bar):
        self.inner, self.bar = inner, bar

    def __getattr__(self, name):
        return getattr(self.inner, name)

    def generate(self, *args, **kwargs):
        out = self.inner.generate(*args, **kwargs)
        self.bar.update(1)
        return out

    def generate_multiple(self, prompts, *args, batch_size=50, **kwargs):
        out = []
        for i in range(0, len(prompts), batch_size):
            chunk = prompts[i:i + batch_size]
            out += self.inner.generate_multiple(chunk, *args, batch_size=batch_size, **kwargs)
            self.bar.update(len(chunk))
        return out


class ProgressJudge:
    """Forwards to a GeminiEvaluator and advances `bar` once per `score_*` call."""

    def __init__(self, inner, bar):
        self.inner, self.bar, self.lock = inner, bar, threading.Lock()

    def __getattr__(self, name):
        attr = getattr(self.inner, name)
        if not name.startswith("score_"):
            return attr

        def scored(*args, **kwargs):
            out = attr(*args, **kwargs)
            with self.lock:  # evaluate_alpaca scores from 30 threads
                self.bar.update(1)
            return out
        return scored


def eval_total(with_alpaca: bool) -> int:
    """Progress-bar steps for one full validation pass (Alpaca = 1 generation + 2 judge calls per item)."""
    n = len(val_mmlu) + len(val_mc["qa"]) + len(val_mc["simdom"])
    return n + 3 * N_ALPACA if with_alpaca else n


def mc_norm(acc: float, base: float) -> float:
    return min(1.0, max((acc - 0.25) / (base - 0.25), 0.0)) if base != 0.25 else 0.0


def mmlu_acc(wrapped) -> float:
    correct, _, _ = evaluate_mmlu_generation(wrapped, val_mmlu)
    return correct / len(val_mmlu)


def mc_acc(wrapped, name: str) -> float:
    correct, _, _ = evaluate_mc_generation(wrapped, val_mc[name])
    return correct / len(val_mc[name])


def alpaca_scores(wrapped, judge_=None) -> tuple:
    instr, flu, _ = evaluate_alpaca(wrapped, judge_ or judge, "train", batch_size=ALPACA_BATCH)
    return float(np.mean(instr)), float(np.mean(flu))


# Baseline = the original model on the validation split (computed once, reused on re-runs).
if "val_base" not in globals() or (TUNE_WITH_ALPACA and "alpaca_instr" not in val_base):
    with tqdm(total=eval_total(TUNE_WITH_ALPACA), desc="baseline (original)", unit="step") as bar:
        m = ProgressModel(MODELS["original"], bar)
        bar.set_postfix_str("MMLU")
        val_base = {"mmlu": mmlu_acc(m)}
        bar.set_postfix_str("concept MC")
        val_base["qa"] = mc_acc(m, "qa")
        bar.set_postfix_str("similar-domain MC")
        val_base["simdom"] = mc_acc(m, "simdom")
        if TUNE_WITH_ALPACA:
            bar.set_postfix_str("AlpacaEval")
            val_base["alpaca_instr"], val_base["alpaca_flu"] = alpaca_scores(m, ProgressJudge(judge, bar))
val_base


In [ ]:
def h_score(wrapped) -> dict:
    """Stage-1 H score of one edited model, with the repo's early-stop gates."""
    row = {"mmlu_acc": mmlu_acc(wrapped)}
    row["mmlu_frac"] = mc_norm(row["mmlu_acc"], val_base["mmlu"])
    if row["mmlu_frac"] < MIN_MMLU:
        return {**row, "H": 0.0, "gate": "mmlu"}

    for name in ("qa", "simdom"):
        row[f"{name}_acc"] = mc_acc(wrapped, name)
        row[f"{name}_frac"] = mc_norm(row[f"{name}_acc"], val_base[name])
    row["efficacy"] = 1.0 - row["qa_frac"]
    row["specificity"] = harmonic_mean([row["mmlu_frac"], row["simdom_frac"]])
    axes = [row["efficacy"], row["specificity"]]

    if TUNE_WITH_ALPACA:
        instr, flu = alpaca_scores(wrapped)
        row["instr_frac"] = min(1.0, instr / val_base["alpaca_instr"])
        row["flu_frac"] = min(1.0, flu / val_base["alpaca_flu"])
        row["coherence"] = harmonic_mean([row["instr_frac"], row["flu_frac"]])
        axes.append(row["coherence"])
        if row["instr_frac"] < MIN_ALPACA or row["flu_frac"] < MIN_ALPACA:
            return {**row, "H": 0.0, "gate": "alpaca"}

    return {**row, "H": harmonic_mean(axes), "gate": ""}


# Fallback for kernels where 3.3.c ran before apply_ember_edit existed (same code as 3.3.c).
if "apply_ember_edit" not in globals():
    edit_idx = torch.tensor(edit_ids, dtype=torch.long)
    with torch.no_grad():
        original_rows = W_orig.index_select(0, edit_idx.to(W_orig.device)).cpu().clone()

    def apply_ember_edit(W: torch.Tensor, delta: float) -> None:
        """Set the edited rows of embedding matrix W to e_t - delta * c_t (always from the original rows)."""
        with torch.no_grad():
            e = original_rows.to(W.device).float() * EMB_SCALE
            e_new = e - delta * C.to(device=W.device, dtype=torch.float32)
            W.index_copy_(0, edit_idx.to(W.device), (e_new / EMB_SCALE).to(W.dtype))


if "tuned_model" not in globals():
    tuned_model = share_all_but_embeddings(original_model)
W_tuned = tuned_model.get_input_embeddings().weight
MODELS["tuned"] = WrappedHFModel(tuned_model, hf_tok)

grid_rows = []
for delta in tqdm(DELTA_GRID, desc="Stage 1: embedding grid"):
    apply_ember_edit(W_tuned, delta)
    with Timer() as t:
        row = h_score(MODELS["tuned"])
    grid_rows.append({"delta": delta, **row, "seconds": round(t["elapsed"], 1)})
    print(f"delta={delta:>6}: H={row['H']:.3f}  {('gated by ' + row['gate']) if row['gate'] else ''}")

grid = pd.DataFrame(grid_rows)
passed = grid[grid["H"] > 0]
# Same rule as io.pick_best_embed_delta: max H, ties -> smaller delta, all zero -> no edit.
BEST_DELTA = 0.0 if passed.empty else float(
    passed.sort_values(["H", "delta"], ascending=[False, True]).iloc[0]["delta"])
apply_ember_edit(W_tuned, BEST_DELTA)

print(f"\nBEST_DELTA = {BEST_DELTA}" + ("  (every H was 0, so no edit)" if passed.empty else ""))
grid


### 3.4.b Compare all three models

`MODELS` now holds `original`, `erased` ($\delta$ = `DELTA`) and `tuned` ($\delta$ = `BEST_DELTA`), so the
3.3.d API covers all of them: `ask(q, model="tuned")`, `compare([...])`, `compare_mc()`.
`compare_mc()` reports on the **test** split, which the grid never saw.


In [ ]:
print(f"erased: delta={DELTA}   tuned: delta={BEST_DELTA}\n")
compare(QUESTIONS_EN + QUESTIONS_SIM)


In [ ]:
mc_table = compare_mc()
mc_table


In [ ]:
# Check if the one feature direction of Gambling belongs to the vector space spanned by the other feature directions
# z_i is in span{z_j : j != i} iff its least-squares projection onto that span leaves a ~0 residual.
# Reference point: with k-1 = 199 directions in d = 4096 dims, a random direction keeps ~sqrt(1 - 199/4096) ≈ 0.975
# of its norm, so a residual near 0.98 means "no more in the span than a random vector".
Z64 = Z.double()
d, k = Z64.shape


def span_residual(z: torch.Tensor, B: torch.Tensor):
    """Relative residual ||z - P_B z|| / ||z|| and the least-squares coefficients of z on the columns of B."""
    coef = torch.linalg.lstsq(B, z.unsqueeze(1), driver="gelsd").solution.squeeze(1)
    return ((z - B @ coef).norm() / z.norm()).item(), coef


rows = []
for i in range(k):
    others = [j for j in range(k) if j != i]
    resid, _ = span_residual(Z64[:, i], Z64[:, others])
    rows.append({"feature": i, "resid_frac": resid, "in_F_C": i in F_C})
span_df = pd.DataFrame(rows).sort_values("resid_frac").reset_index(drop=True)

g = torch.Generator().manual_seed(SEED)
rand_resid = [span_residual(torch.randn(d, generator=g, dtype=torch.float64), Z64[:, 1:])[0] for _ in range(200)]

print(f"rank(Z) = {torch.linalg.matrix_rank(Z64).item()} / {k}   cond(Z) = {torch.linalg.cond(Z64).item():.1f}")
print(f"random-direction residual: mean={np.mean(rand_resid):.4f}  min={np.min(rand_resid):.4f}")
print(f"other features' residual  : median={span_df.loc[~span_df.in_F_C, 'resid_frac'].median():.4f}  "
      f"min={span_df.loc[~span_df.in_F_C, 'resid_frac'].min():.4f}\n")

for feat in F_C:
    others = [j for j in range(k) if j != feat]
    resid, coef = span_residual(Z64[:, feat], Z64[:, others])
    rank_pos = int(span_df.index[span_df.feature == feat][0]) + 1
    verdict = "IN the span" if resid < 1e-6 else ("mostly in the span" if resid < 0.5 else "NOT in the span")
    print(f"feature {feat}: residual = {resid:.4f}  (||proj|| / ||z|| = {np.sqrt(max(0.0, 1 - resid**2)):.4f}), "
          f"rank {rank_pos}/{k} from most-in-span  ->  {verdict}")

    # Which other features carry the projection (|coef_j| * ||z_j|| = size of each term in the combination).
    contrib = (coef.abs() * Z64[:, others].norm(dim=0)) / Z64[:, feat].norm()
    top = torch.argsort(contrib, descending=True)[:5]
    print("  largest terms: " + ", ".join(f"z_{others[t]} ({contrib[t]:.3f})" for t in top.tolist()))

span_df.head(10)


In [ ]:
print(Z.size())
print(Y.size())
print(RANK)

In [ ]:
# Please fill here

In [ ]:
# For each of token the "Gambling Concept" direction appears on - how many feature are activated in the factorization?
# Support of feature i = tokens t with Y_{i,t} != 0. For each such token, count the nonzero entries of its column Y_{:,t}
# (the concept feature itself included) and compare with every token in V*.
# share = |Y_{i,t}| ||z_i|| / sum_j |Y_{j,t}| ||z_j||: how much of the token's feature reconstruction the concept feature carries.
#
# proj = <e_t, z_i / ||z_i||>, the scalar projection on the concept direction (z oriented so its tokens have Y > 0),
# in three normalizations:
#   / avg feature : proj / mean_j |<e_t, z_j / ||z_j||>| over all k feature directions (z_i included)
#   / random dir  : proj / E_u |<e_t, u>| for u uniform on the unit sphere. Exact, no sampling:
#                   E|<e, u>| = ||e|| * Gamma(d/2) / (sqrt(pi) * Gamma((d+1)/2))  (~ ||e|| sqrt(2 / (pi d)))
#   z vs V*       : (proj - mean_s proj_s) / std_s proj_s over all tokens s in V*, i.e. the same direction across tokens
from collections import Counter

active = Y.abs() > 0                     # (k, |V*|)
n_active = active.sum(0).numpy()         # active features per token
z_norms = Z.norm(dim=0)                  # (k,)
rho = stats.set_index("feature")["ratio_abs"]

E_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).float().cpu()  # E_vstar is deleted in 3.3.a
proj_all = E_ref @ (Z / z_norms)         # (|V*|, k): projection of every token on every unit feature direction
avg_feat_proj = proj_all.abs().mean(1)   # (|V*|,)
d = Z.shape[0]
rand_proj = E_ref.norm(dim=1) * math.exp(math.lgamma(d / 2) - math.lgamma((d + 1) / 2)) / math.sqrt(math.pi)

for feat in F_C:
    support = torch.nonzero(active[feat]).flatten()
    support = support[torch.argsort(Y[feat, support].abs(), descending=True)].tolist()
    counts = n_active[support]

    sign = 1.0 if Y[feat, support].sum() >= 0 else -1.0
    proj = sign * proj_all[:, feat]
    proj_z = (proj - proj.mean()) / proj.std()

    print(f"=== feature {feat}: on {len(support)} tokens ===")
    print(f"proj on z_{feat} over all of V*: mean={proj.mean():.3f}  std={proj.std():.3f}")
    print(f"features per token: mean={counts.mean():.2f} (all of V*: {n_active.mean():.2f})  "
          f"min={counts.min()}  max={counts.max()}  only this feature: {int((counts == 1).sum())}")
    dist = pd.DataFrame({
        f"feature {feat} tokens": pd.Series(counts).value_counts(normalize=True),
        "all V*": pd.Series(n_active).value_counts(normalize=True),
    }).fillna(0.0).sort_index()
    dist.index.name = "# active features"
    display(dist.style.format("{:.1%}"))

    rows, co = [], Counter()
    for j in support:
        terms = Y[:, j].abs() * z_norms
        others = [i for i in torch.nonzero(active[:, j]).flatten().tolist() if i != feat]
        others.sort(key=lambda i: -terms[i].item())
        co.update(others)
        rows.append({
            "token": tokenizer.convert_ids_to_tokens(vstar_ids[j]),
            "label": LABEL_TAG[codes[j]],
            f"Y[{feat}]": Y[feat, j].item(),
            "n_active": int(n_active[j]),
            "share": (terms[feat] / terms.sum()).item(),
            "proj": proj[j].item(),
            "proj / avg feature": (proj[j] / avg_feat_proj[j]).item(),
            "proj / random dir": (proj[j] / rand_proj[j]).item(),
            "proj z vs V*": proj_z[j].item(),
            "other features (Y)": ", ".join(f"{i} ({Y[i, j].item():+.2f})" for i in others),
        })
    per_token = pd.DataFrame(rows)
    per_token.index += 1
    with pd.option_context("display.max_rows", None):
        display(per_token)

    # Which other features co-occur most often on this feature's tokens.
    co_df = pd.DataFrame([
        {"feature": i, "shared tokens": n, "frac of support": n / len(support),
         "rho": rho.get(i, float("nan")), "in_F_C": i in F_C}
        for i, n in co.most_common(10)
    ])
    print(f"most frequent co-active features on feature {feat}'s tokens:")
    display(co_df)

In [ ]:
# For each of token the "Gambling Concept" direction appears on - how many feature are activated in the factorization?
# Support of feature i = tokens t with Y_{i,t} != 0. For each such token, count the nonzero entries of its column Y_{:,t}
# (the concept feature itself included) and compare with every token in V*.
# share = |Y_{i,t}| ||z_i|| / sum_j |Y_{j,t}| ||z_j||: how much of the token's feature reconstruction the concept feature carries.
#
# proj = <e_t, z_i / ||z_i||>, the scalar projection on the concept direction (z oriented so its tokens have Y > 0),
# in three normalizations:
#   / avg feature : proj / mean_j |<e_t, z_j / ||z_j||>| over all k feature directions (z_i included)
#   / random dir  : proj / E_u |<e_t, u>| for u uniform on the unit sphere. Exact, no sampling:
#                   E|<e, u>| = ||e|| * Gamma(d/2) / (sqrt(pi) * Gamma((d+1)/2))  (~ ||e|| sqrt(2 / (pi d)))
#   z vs V*       : (proj - mean_s proj_s) / std_s proj_s over all tokens s in V*, i.e. the same direction across tokens
from collections import Counter

active = Y.abs() > 0                     # (k, |V*|)
n_active = active.sum(0).numpy()         # active features per token
z_norms = Z.norm(dim=0)                  # (k,)
rho = stats.set_index("feature")["ratio_abs"]

E_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).float().cpu()  # E_vstar is deleted in 3.3.a
proj_all = E_ref @ (Z / z_norms)         # (|V*|, k): projection of every token on every unit feature direction
avg_feat_proj = proj_all.abs().mean(1)   # (|V*|,)
d = Z.shape[0]
rand_proj = E_ref.norm(dim=1) * math.exp(math.lgamma(d / 2) - math.lgamma((d + 1) / 2)) / math.sqrt(math.pi)

for feat in F_C:
    support = torch.nonzero(active[feat]).flatten()
    support = support[torch.argsort(Y[feat, support].abs(), descending=True)].tolist()
    counts = n_active[support]

    sign = 1.0 if Y[feat, support].sum() >= 0 else -1.0
    proj = sign * proj_all[:, feat]
    proj_z = (proj - proj.mean()) / proj.std()

    print(f"=== feature {feat}: on {len(support)} tokens ===")
    print(f"proj on z_{feat} over all of V*: mean={proj.mean():.3f}  std={proj.std():.3f}")
    print(f"features per token: mean={counts.mean():.2f} (all of V*: {n_active.mean():.2f})  "
          f"min={counts.min()}  max={counts.max()}  only this feature: {int((counts == 1).sum())}")
    dist = pd.DataFrame({
        f"feature {feat} tokens": pd.Series(counts).value_counts(normalize=True),
        "all V*": pd.Series(n_active).value_counts(normalize=True),
    }).fillna(0.0).sort_index()
    dist.index.name = "# active features"
    display(dist.style.format("{:.1%}"))

    rows, co = [], Counter()
    for j in support:
        terms = Y[:, j].abs() * z_norms
        others = [i for i in torch.nonzero(active[:, j]).flatten().tolist() if i != feat]
        others.sort(key=lambda i: -terms[i].item())
        co.update(others)
        rows.append({
            "token": tokenizer.convert_ids_to_tokens(vstar_ids[j]),
            "label": LABEL_TAG[codes[j]],
            f"Y[{feat}]": Y[feat, j].item(),
            "n_active": int(n_active[j]),
            "share": (terms[feat] / terms.sum()).item(),
            "proj": proj[j].item(),
            "proj / avg feature": (proj[j] / avg_feat_proj[j]).item(),
            "proj / random dir": (proj[j] / rand_proj[j]).item(),
            "proj z vs V*": proj_z[j].item(),
            "other features (Y)": ", ".join(f"{i} ({Y[i, j].item():+.2f})" for i in others),
        })
    per_token = pd.DataFrame(rows)
    per_token.index += 1
    with pd.option_context("display.max_rows", None):
        display(per_token)

    # Which other features co-occur most often on this feature's tokens.
    co_df = pd.DataFrame([
        {"feature": i, "shared tokens": n, "frac of support": n / len(support),
         "rho": rho.get(i, float("nan")), "in_F_C": i in F_C}
        for i, n in co.most_common(10)
    ])
    print(f"most frequent co-active features on feature {feat}'s tokens:")
    display(co_df)

In [ ]:
# All tokens in V*, sorted by proj / avg feature (descending). Uses the variables computed in the cell above.
SHOW_ROWS = 100  # None shows all |V*| rows

terms_all = Y.abs() * z_norms[:, None]            # (k, |V*|): size of each term |Y_{j,t}| ||z_j||
token_strs = tokenizer.convert_ids_to_tokens(vstar_ids)

for feat in F_C:
    sign = 1.0 if Y[feat][active[feat]].sum() >= 0 else -1.0
    proj = sign * proj_all[:, feat]
    proj_z = (proj - proj.mean()) / proj.std()

    all_tokens = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "in_support": active[feat].numpy(),
        # f"Y[{feat}]": Y[feat].numpy(),
        # "n_active": n_active,
        "share": (terms_all[feat] / terms_all.sum(0)).numpy(),
        # "proj": proj.numpy(),
        "proj / avg feature": (proj / avg_feat_proj).numpy(),
        # "proj / random dir": (proj / rand_proj).numpy(),
        "proj z vs V*": proj_z.numpy(),
    }).sort_values("proj / avg feature", ascending=False, ignore_index=True)
    all_tokens.index += 1

    n_sup = int(active[feat].sum())
    print(f"=== feature {feat}: all {len(all_tokens)} tokens of V*, sorted by proj / avg feature ===")
    # print(f"support tokens among the top {n_sup}: {int(all_tokens['in_support'].head(n_sup).sum())} / {n_sup}")
    with pd.option_context("display.max_rows", None):
        display(all_tokens if SHOW_ROWS is None else all_tokens.head(SHOW_ROWS))